
## **1. Imports**


In [1]:
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Sentence-BERT import OK")

C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\transformers\utils\generic.py:500: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\transformers\utils\generic.py:357: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _register_pytree_node(
C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\transformers\utils\generic.py:357: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_p

transformers: 4.52.4
sentence-transformers: 5.1.2


C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\transformers\utils\generic.py:357: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _register_pytree_node(


Sentence-BERT import OK


In [2]:
!pip install pandas requests tqdm openpyxl

# 2. Claude Opus 4.6

### F1&Accuracy by template

In [3]:
import pandas as pd
import re

# INPUT = "1_1_with_answers_recovered.csv"
# INPUT = "1_1_with_answers_claude-opus_4_6.csv"
INPUT = "1_1_with_answers_claude-opus_4_6.csv"

COL_TEMPLATE = "template_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

from pathlib import Path
SKIP_CLAUDE = not Path(INPUT).exists()
if SKIP_CLAUDE:
    print(f"Claude Opus: missing {INPUT}. Cell skipped. Dictionary not saved.")
    df = pd.DataFrame(columns=[COL_TEMPLATE, COL_GT, COL_PRED, "bool_model_answer"])
else:
    df = pd.read_csv(INPUT)


def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t


def extract_yes_no_from_text(text: str) -> str:
    s = "" if pd.isna(text) else str(text)
    matches = re.findall(r"@\s*(YES|NO)\b", s, flags=re.IGNORECASE)
    if matches:
        return matches[-1].lower()
    t = normalize_text(s)
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    return ""


if not SKIP_CLAUDE:
    if "bool_model_answer" not in df.columns:
        df["bool_model_answer"] = df[COL_PRED].apply(extract_yes_no_from_text)
    else:
        # keep existing column but normalize values
        df["bool_model_answer"] = df["bool_model_answer"].apply(normalize_label)
        missing_mask = df["bool_model_answer"].isin(["", "nan", "none", "null"])
        df.loc[missing_mask, "bool_model_answer"] = df.loc[missing_mask, COL_PRED].apply(extract_yes_no_from_text)

df["gt_norm"] = df[COL_GT].apply(normalize_label)
pred_norm = df["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
pred_norm = pred_norm.where(pred_norm.isin(["yes", "no"]), "")
df["bool_model_answer"] = pred_norm
df["is_correct"] = df["bool_model_answer"] == df["gt_norm"]


# ---- metrics per each template_id ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({
        "n": n,
        "accuracy": acc,
        "f1": f1,
        "f1_type": f1_type,
    })


rows = []
if SKIP_CLAUDE:
    report = pd.DataFrame(columns=["n", "accuracy", "f1", "f1_type", COL_TEMPLATE])
else:
    for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
        m = group_metrics(g)
        m[COL_TEMPLATE] = template_id
        rows.append(m)
    report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)
report


,n,accuracy,f1,f1_type,template_id
0,154,0.785714,0.845070,binary(yes),1
1,187,0.561497,0.594059,binary(yes),2
2,1,0.000000,0.000000,macro(GT-classes),3
3,38,0.000000,0.000000,macro(GT-classes),4
4,6,0.000000,0.000000,macro(GT-classes),5
5,10,0.000000,0.000000,macro(GT-classes),7
6,24,1.000000,1.000000,binary(yes),8
7,18,0.000000,0.000000,macro(GT-classes),9
8,10,0.000000,0.000000,macro(GT-classes),10
9,8,0.000000,0.000000,macro(GT-classes),12


### Imports and config

In [4]:
import numpy as np
import pandas as pd
import re
from sklearn.metrics import f1_score

# config
MODEL_NAME = "Claude Opus"

# Column config.
# COL_GT is resolved against candidates after dataframe preparation.
COL_GT = "ground_truth"
COL_GT_CANDIDATES = ["ground_truth_answer"]
COL_PRED = "model_answer"

if "SKIP_CLAUDE" not in globals():
    SKIP_CLAUDE = False


### Prepare basic dataframe and normalise values

In [5]:
def _ensure_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()

    if "bool_model_answer" not in df_local.columns:
        matches = df_local["model_answer"].fillna("").astype(str).str.findall(
            r"@\s*(YES|NO)\b", flags=re.IGNORECASE
        )
        df_local["bool_model_answer"] = matches.apply(lambda m: m[-1].lower() if m else "")
    else:
        b = df_local["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
        b = b.where(b.isin(["yes", "no"]), "")
        df_local["bool_model_answer"] = b

    if "relation_type" in df_local.columns:
        rel = df_local["relation_type"].fillna("").astype(str).str.strip()
        rel = rel.where(~rel.str.lower().isin(["connectivity", "not_direct_connectivity"]), "connectivity")
        df_local["relation_type"] = rel
        # Build normalized relation type AFTER remapping.
        df_local["_relation_type_norm"] = rel.str.lower()
    else:
        df_local["_relation_type_norm"] = ""

    return df_local


df = _ensure_columns(df)

# Resolve GT column robustly.
if COL_GT not in df.columns:
    for cand in COL_GT_CANDIDATES:
        if cand in df.columns:
            COL_GT = cand
            break
if COL_GT not in df.columns:
    raise KeyError(
        f"GT column not found. Tried {COL_GT_CANDIDATES}. Available columns: {list(df.columns)}"
    )
if COL_PRED not in df.columns:
    raise KeyError(
        f"Prediction column '{COL_PRED}' not found. Available columns: {list(df.columns)}"
    )

print(f"Resolved columns -> COL_GT: {COL_GT}, COL_PRED: {COL_PRED}")
print(df.shape)
display(df.head(3))


Resolved columns -> COL_GT: ground_truth_answer, COL_PRED: model_answer
(456, 23)


,generated_question_id,template_id,layer_id,scene_id,file_path,question_text,object_type_filled,feature_name_filled,ground_truth_answer,answer_type,...,dimension_source_view,model_answer,openrouter_request_id,error_reason,row_index,bool_model_answer,model,gt_norm,is_correct,_relation_type_norm
0,0,1,1_1_1,1,C:\Users\ritaMZ\WebstormProjects\thesis\backen...,"Does this view contain a handrails? \nAnswer ""...",handrails,NaN,yes,bool,...,NaN,The floor plan shows a toilet room (1D26) with...,row_0,NaN,0.0,yes,anthropic/claude-opus-4.6,yes,True,
1,1,1,1_1_1,1,C:\Users\ritaMZ\WebstormProjects\thesis\backen...,"Does this view contain a toilet? \nAnswer ""yes...",toilet,NaN,yes,bool,...,NaN,"The floor plan shows a room labeled ""TOILET 1D...",row_1,NaN,1.0,yes,anthropic/claude-opus-4.6,yes,True,
2,2,1,1_1_1,1,C:\Users\ritaMZ\WebstormProjects\thesis\backen...,"Does this view contain a sink? \nAnswer ""yes"" ...",sink,NaN,yes,bool,...,NaN,"The floor plan shows a toilet room (labeled ""T...",row_2,NaN,2.0,yes,anthropic/claude-opus-4.6,yes,True,


### Helper functions

In [6]:
import numpy as np
import re

df_eval = df.copy()

# -------- Normalize routing columns --------
df_eval["_template_id_norm"] = pd.to_numeric(df_eval.get("template_id"), errors="coerce").astype("Int64")
layer_source = "layer_1" if "layer_1" in df_eval.columns else ("layer_id" if "layer_id" in df_eval.columns else None)
if layer_source is None:
    df_eval["_layer_1_norm"] = ""
else:
    df_eval["_layer_1_norm"] = (
        df_eval[layer_source]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(".", "_", regex=False)
    )
df_eval["_answer_type_norm"] = df_eval.get("answer_type", "").fillna("").astype(str).str.strip().str.lower()

# Keep bool extraction robust for cases where bool_model_answer is empty.
if "bool_model_answer" not in df_eval.columns:
    df_eval["bool_model_answer"] = ""


def _norm_bool(v) -> str:
    s = "" if pd.isna(v) else str(v).strip().lower()
    if s in {"yes", "y", "true", "1"}:
        return "yes"
    if s in {"no", "n", "false", "0"}:
        return "no"
    return ""


def _extract_bool_from_pred(v) -> str:
    s = "" if pd.isna(v) else str(v)
    m = re.findall(r"@\s*(YES|NO)\b", s, flags=re.IGNORECASE)
    if m:
        return m[-1].lower()
    return _norm_bool(s)

def _extract_answer_after_at(v) -> str:
    # Extract the final answer after the last "@" marker.
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""

    if "@" in s:
        s = s.rsplit("@", 1)[-1].strip()

    # Keep only the first line after "@"
    s = s.splitlines()[0].strip() if s else ""

    # Remove wrapping punctuation
    s = s.strip(" \t\n\r-вЂ“вЂ”:;,.\"'`()[]{}")

    return s
df_eval["parsed_model_answer"] = df_eval[COL_PRED].apply(_extract_answer_after_at)

def _binary_eval_view(sub: pd.DataFrame) -> pd.DataFrame:
    if sub.empty:
        return sub.copy()
    out = sub.copy()
    if "gt_norm" in out.columns:
        gt = out["gt_norm"].map(_norm_bool)
    else:
        gt = out[COL_GT].map(_norm_bool)
    pred = out["bool_model_answer"].map(_norm_bool)
    pred = pred.where(pred != "", out[COL_PRED].map(_extract_bool_from_pred))
    out["_gt_bool"] = gt
    out["_pred_bool"] = pred
    out = out[out["_gt_bool"].isin(["yes", "no"]) & out["_pred_bool"].isin(["yes", "no"])].copy()
    return out



def _binary_metrics(sub: pd.DataFrame) -> dict:
    cols = {
        "accuracy_binary": np.nan,
        "f1_macro": np.nan,
        "f1_yes": np.nan,
        "f1_no": np.nan,
        "majority_baseline": np.nan,
        "baseline_f1_macro": np.nan,
        "baseline_f1_yes": np.nan,
        "baseline_f1_no": np.nan,
    }
    ev = _binary_eval_view(sub)
    if ev.empty:
        return cols

    y_true = ev["_gt_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
    y_pred = ev["_pred_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
    acc = float((y_true == y_pred).mean())

    from sklearn.metrics import f1_score
    f1_macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_yes = float(f1_score(y_true, y_pred, pos_label=1, zero_division=0))
    f1_no = float(f1_score(y_true, y_pred, pos_label=0, zero_division=0))

    dist = ev["_gt_bool"].value_counts()
    maj = dist.idxmax()
    y_base = np.ones_like(y_true) if maj == "yes" else np.zeros_like(y_true)
    majority_acc = float((y_true == y_base).mean())
    f1_macro_b = float(f1_score(y_true, y_base, average="macro", zero_division=0))
    f1_yes_b = float(f1_score(y_true, y_base, pos_label=1, zero_division=0))
    f1_no_b = float(f1_score(y_true, y_base, pos_label=0, zero_division=0))

    return {
        "accuracy_binary": acc,
        "f1_macro": f1_macro,
        "f1_yes": f1_yes,
        "f1_no": f1_no,
        "majority_baseline": majority_acc,
        "baseline_f1_macro": f1_macro_b,
        "baseline_f1_yes": f1_yes_b,
        "baseline_f1_no": f1_no_b,
    }


def _binary_distribution(sub: pd.DataFrame) -> dict:
    if sub.empty:
        return {"n": 0, "yes": 0, "no": 0}
    if "gt_norm" in sub.columns:
        gt = sub["gt_norm"].map(_norm_bool)
    else:
        gt = sub[COL_GT].map(_norm_bool)
    gt = gt[gt.isin(["yes", "no"])]
    return {
        "n": int(len(gt)),
        "yes": int((gt == "yes").sum()),
        "no": int((gt == "no").sum()),
    }


def _norm_text(v) -> str:
    s = "" if pd.isna(v) else str(v)
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def _valid_text_pairs(sub: pd.DataFrame):
    if sub.empty:
        return pd.Series(dtype=str), pd.Series(dtype=str)

    gt = sub[COL_GT].map(_norm_text)

    # Use parsed answer after "@" for MCQ and text-based metrics
    pred_col = "parsed_model_answer" if "parsed_model_answer" in sub.columns else COL_PRED
    pred = sub[pred_col].map(_norm_text)

    valid = gt != ""
    return gt[valid], pred[valid]


def _exact_match(sub: pd.DataFrame) -> float:
    gt, pred = _valid_text_pairs(sub)
    if len(gt) == 0:
        return np.nan
    return float((gt.values == pred.values).mean())


def _tokenize(s: str):
    return re.findall(r"\w+", s.lower(), flags=re.UNICODE)


def _token_f1_single(gt: str, pred: str) -> float:
    gt_t = _tokenize(gt)
    pr_t = _tokenize(pred)
    if len(gt_t) == 0 and len(pr_t) == 0:
        return 1.0
    if len(gt_t) == 0 or len(pr_t) == 0:
        return 0.0
    from collections import Counter
    c_gt = Counter(gt_t)
    c_pr = Counter(pr_t)
    overlap = sum((c_gt & c_pr).values())
    if overlap == 0:
        return 0.0
    p = overlap / len(pr_t)
    r = overlap / len(gt_t)
    return 2 * p * r / (p + r)


def _macro_token_f1(sub: pd.DataFrame) -> float:
    gt, pred = _valid_text_pairs(sub)
    if len(gt) == 0:
        return np.nan
    vals = [_token_f1_single(g, p) for g, p in zip(gt.tolist(), pred.tolist())]
    return float(np.mean(vals)) if vals else np.nan


def _accuracy_mcq(sub: pd.DataFrame) -> float:
    return _exact_match(sub)


def _mean_sbert_similarity(sub: pd.DataFrame) -> float:
    # Compute mean Sentence-BERT similarity for valid gold/prediction text pairs.
    gt, pred = _valid_text_pairs(sub)
    if len(gt) == 0:
        return np.nan

    try:
        from sentence_transformers import SentenceTransformer, util
    except Exception as e:
        print(f"Sentence-BERT import failed: {e}")
        return np.nan

    try:
        model_sbert = globals().get("_SCENE_SBERT_MODEL")
        if model_sbert is None:
            model_sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            globals()["_SCENE_SBERT_MODEL"] = model_sbert

        emb_gt = model_sbert.encode(
            gt.tolist(),
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        emb_pr = model_sbert.encode(
            pred.tolist(),
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        scores = util.pairwise_dot_score(emb_gt, emb_pr).cpu().numpy()
        return float(np.mean(scores))

    except Exception as e:
        print(f"Sentence-BERT similarity failed: {e}")
        return np.nan
    return float(np.mean(scores))

def _debug_text_pairs(sub: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    # Show a few normalized gold/pred pairs used in text-based metrics.
    gt, pred = _valid_text_pairs(sub)
    out = pd.DataFrame({
        "gold": gt.tolist(),
        "pred": pred.tolist(),
    })
    return out.head(n)

def _taskwise_overall_accuracy(sub: pd.DataFrame, bool_mask: pd.Series, mcq_mask: pd.Series, text_mask: pd.Series) -> float:
    correct_vals = []

    ev_bool = _binary_eval_view(sub[bool_mask])
    if not ev_bool.empty:
        correct_vals.extend((ev_bool["_gt_bool"] == ev_bool["_pred_bool"]).astype(float).tolist())

    gt_mcq, pred_mcq = _valid_text_pairs(sub[mcq_mask])
    if len(gt_mcq) > 0:
        correct_vals.extend((gt_mcq.values == pred_mcq.values).astype(float).tolist())

    gt_txt, pred_txt = _valid_text_pairs(sub[text_mask])
    if len(gt_txt) > 0:
        correct_vals.extend((gt_txt.values == pred_txt.values).astype(float).tolist())

    if len(correct_vals) == 0:
        return np.nan
    return float(np.mean(correct_vals))


def _make_excel_table(table_name: str, ordered_metrics: list[tuple], dataset_characteristics: dict | str | None = None) -> pd.DataFrame:
    def _fmt_ds(ds_obj):
        if ds_obj is None:
            return ""
        if isinstance(ds_obj, str):
            return ds_obj
        if isinstance(ds_obj, dict):
            return ", ".join([f"{k}={v}" for k, v in ds_obj.items()])
        return str(ds_obj)

    default_ds = _fmt_ds(dataset_characteristics)
    rows = []
    for item in ordered_metrics:
        if len(item) == 2:
            metric_name, metric_value = item
            ds = default_ds
        elif len(item) == 3:
            metric_name, metric_value, metric_ds = item
            ds = _fmt_ds(metric_ds)
        else:
            raise ValueError(f"ordered_metrics items must have length 2 or 3, got: {item}")
        rows.append({
            "Table": table_name,
            "Metric": metric_name,
            "Value": metric_value,
            "Dataset characteristics": ds,
        })
    return pd.DataFrame(rows)


# -------- Routing masks (no relation_type grouping) --------
ocr_mask = df_eval["_layer_1_norm"] == "1_1_2"
object_mask = (df_eval["_template_id_norm"] == 1) & (~ocr_mask)
feature_mask = df_eval["_template_id_norm"].isin([2, 4, 5, 7]) & (~ocr_mask)

feature_binary_mask = feature_mask & (df_eval["_template_id_norm"] == 2)
feature_text_mask = feature_mask & df_eval["_template_id_norm"].isin([4, 5, 7])
feature_mcq_mask = feature_mask & (~feature_text_mask) & df_eval["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])

ocr_bool_mask = ocr_mask & (df_eval["_answer_type_norm"] == "bool")
ocr_text_numeric_mask = ocr_mask & df_eval["_answer_type_norm"].isin(["numeric", "text"])

df_object = df_eval[object_mask].copy()
df_feature = df_eval[feature_mask].copy()
df_ocr = df_eval[ocr_mask].copy()

overall_idx = df_object.index.union(df_feature.index).union(df_ocr.index)
df_scene_overall = df_eval.loc[overall_idx].copy()


In [7]:
# -------- Scene perception (object) --------
object_binary_metrics = _binary_metrics(df_object)
object_dist = _binary_distribution(df_object)

object_ds_overall = {"n": int(len(df_object))}
object_ds_binary = {
    "n": int(object_dist["n"]),
    "yes": object_dist["yes"],
    "no": object_dist["no"],
}
object_ds_mcq = {"n": 0}
object_ds_text_numeric = {"n": 0}

object_metrics = {
    "overall accuracy": object_binary_metrics["accuracy_binary"],
    "accuracy_mcq": np.nan,
    "binary tasks": object_binary_metrics,
    "dataset characteristics": {
        "overall": object_ds_overall,
        "binary": object_ds_binary,
        "mcq": object_ds_mcq,
        "text_numeric": object_ds_text_numeric,
    },
}

scene_perception_object_table = _make_excel_table(
    "Scene perception (object)",
    [
        ("overall accuracy", object_metrics["overall accuracy"], object_ds_overall),
        ("accuracy_mcq", np.nan, object_ds_mcq),
        ("binary_tasks/accuracy_binary", object_binary_metrics["accuracy_binary"], object_ds_binary),
        ("binary_tasks/f1_macro", object_binary_metrics["f1_macro"], object_ds_binary),
        ("binary_tasks/f1_yes", object_binary_metrics["f1_yes"], object_ds_binary),
        ("binary_tasks/f1_no", object_binary_metrics["f1_no"], object_ds_binary),
        ("binary_tasks/majority_baseline", object_binary_metrics["majority_baseline"], object_ds_binary),
        ("binary_tasks/baseline_f1_macro", object_binary_metrics["baseline_f1_macro"], object_ds_binary),
        ("binary_tasks/baseline_f1_yes", object_binary_metrics["baseline_f1_yes"], object_ds_binary),
        ("binary_tasks/baseline_f1_no", object_binary_metrics["baseline_f1_no"], object_ds_binary),
    ],
)

In [8]:
# -------- Scene perception (feature) --------
feature_binary_df = df_eval[feature_binary_mask].copy()
feature_text_df = df_eval[feature_text_mask].copy()

feature_binary_metrics = _binary_metrics(feature_binary_df)
feature_binary_dist = _binary_distribution(feature_binary_df)
feature_exact_match = _exact_match(feature_text_df)
feature_token_f1 = _macro_token_f1(feature_text_df)
feature_sbert = _mean_sbert_similarity(feature_text_df)
feature_overall_accuracy = _taskwise_overall_accuracy(df_feature, feature_binary_mask.loc[df_feature.index], feature_mcq_mask.loc[df_feature.index], feature_text_mask.loc[df_feature.index])

feature_n_overall = int(len(df_feature))
feature_n_binary = int(feature_binary_dist["n"])
feature_n_text_numeric = int(len(_valid_text_pairs(feature_text_df)[0]))

feature_ds_overall = {"n": feature_n_overall}
feature_ds_binary = {"n": feature_n_binary, "yes": feature_binary_dist["yes"], "no": feature_binary_dist["no"]}
feature_ds_text_numeric = {"n": feature_n_text_numeric}

feature_metrics = {
    "overall accuracy": feature_overall_accuracy,
    "Sentence-BERT similarity": feature_sbert,
    "Macro-averaged F1 score (token-based)": feature_token_f1,
    "Exact Match": feature_exact_match,
    "binary tasks": feature_binary_metrics,
    "dataset characteristics": {
        "overall": feature_ds_overall,
        "binary": feature_ds_binary,
        "text_numeric": feature_ds_text_numeric,
    },
}
scene_perception_feature_table = _make_excel_table(
    "Scene perception (feature)",
    [
        ("overall accuracy", feature_overall_accuracy, feature_ds_overall),
        ("Sentence-BERT similarity", feature_sbert, feature_ds_text_numeric),
        ("Macro-averaged F1 score (token-based)", feature_token_f1, feature_ds_text_numeric),
        ("Exact Match", feature_exact_match, feature_ds_text_numeric),
        ("binary_tasks/f1_macro", feature_binary_metrics["f1_macro"], feature_ds_binary),
        ("binary_tasks/f1_yes", feature_binary_metrics["f1_yes"], feature_ds_binary),
        ("binary_tasks/f1_no", feature_binary_metrics["f1_no"], feature_ds_binary),
        ("binary_tasks/majority_baseline", feature_binary_metrics["majority_baseline"], feature_ds_binary),
        ("binary_tasks/baseline_f1_macro", feature_binary_metrics["baseline_f1_macro"], feature_ds_binary),
        ("binary_tasks/baseline_f1_yes", feature_binary_metrics["baseline_f1_yes"], feature_ds_binary),
        ("binary_tasks/baseline_f1_no", feature_binary_metrics["baseline_f1_no"], feature_ds_binary),
    ],
    dataset_characteristics=None,
)



In [9]:
# -------- Scene perception (OCR) --------
ocr_bool_df = df_eval[ocr_bool_mask].copy()
ocr_text_num_df = df_eval[ocr_text_numeric_mask].copy()

ocr_binary_metrics = _binary_metrics(ocr_bool_df)
ocr_exact_match = _exact_match(ocr_text_num_df)
ocr_token_f1 = _macro_token_f1(ocr_text_num_df)
ocr_sbert = _mean_sbert_similarity(ocr_text_num_df)

ocr_binary_dist = _binary_distribution(ocr_bool_df)
ocr_n_binary = int(ocr_binary_dist["n"])
ocr_n_text_numeric = int(len(_valid_text_pairs(ocr_text_num_df)[0]))
ocr_n_mcq = int(len(_valid_text_pairs(df_ocr[df_ocr["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])])[0]))

ocr_ds_binary = {"n": ocr_n_binary, "yes": ocr_binary_dist["yes"], "no": ocr_binary_dist["no"]}
ocr_ds_text_numeric = {"n": ocr_n_text_numeric}
ocr_ds_mcq = {"n": ocr_n_mcq}

ocr_metrics = {
    "Exact Match": ocr_exact_match,
    "Sentence-BERT similarity": ocr_sbert,
    "Macro-averaged F1 score": ocr_token_f1,
    "binary tasks": ocr_binary_metrics,
    "dataset characteristics": {
        "binary": ocr_ds_binary,
        "text_numeric": ocr_ds_text_numeric,
        "mcq": ocr_ds_mcq,
    },
}
scene_perception_ocr_table = _make_excel_table(
    "Scene perception (OCR)",
    [
        ("Exact Match", ocr_exact_match, ocr_ds_text_numeric),
        ("Sentence-BERT similarity", ocr_sbert, ocr_ds_text_numeric),
        ("Macro-averaged F1 score", ocr_token_f1, ocr_ds_text_numeric),
        ("accuracy_binary", ocr_binary_metrics["accuracy_binary"], ocr_ds_binary),
        ("binary_tasks/f1_macro", ocr_binary_metrics["f1_macro"], ocr_ds_binary),
        ("binary_tasks/f1_yes", ocr_binary_metrics["f1_yes"], ocr_ds_binary),
        ("binary_tasks/f1_no", ocr_binary_metrics["f1_no"], ocr_ds_binary),
        ("binary_tasks/majority_baseline", ocr_binary_metrics["majority_baseline"], ocr_ds_binary),
        ("binary_tasks/baseline_f1_macro", ocr_binary_metrics["baseline_f1_macro"], ocr_ds_binary),
        ("binary_tasks/baseline_f1_yes", ocr_binary_metrics["baseline_f1_yes"], ocr_ds_binary),
        ("binary_tasks/baseline_f1_no", ocr_binary_metrics["baseline_f1_no"], ocr_ds_binary),
    ],
    dataset_characteristics=None,
)



In [10]:
# --------  Overall (Scene perception) --------
overall_bool_mask = df_scene_overall["_answer_type_norm"] == "bool"
overall_feature_text_templates_mask = df_scene_overall["_template_id_norm"].isin([4, 5, 7])
overall_mcq_mask = (~overall_bool_mask) & (~overall_feature_text_templates_mask) & (
    df_scene_overall["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])
)
overall_text_numeric_mask = (~overall_bool_mask) & (
    overall_feature_text_templates_mask
    | df_scene_overall["_answer_type_norm"].isin(["text", "numeric"])
)

overall_binary_metrics = _binary_metrics(df_scene_overall[overall_bool_mask])
overall_accuracy_mcq = _accuracy_mcq(df_scene_overall[overall_mcq_mask])
overall_exact_match = _exact_match(df_scene_overall[overall_text_numeric_mask])
overall_token_f1 = _macro_token_f1(df_scene_overall[overall_text_numeric_mask])
overall_sbert = _mean_sbert_similarity(df_scene_overall[overall_text_numeric_mask])
overall_accuracy = _taskwise_overall_accuracy(df_scene_overall, overall_bool_mask, overall_mcq_mask, overall_text_numeric_mask)

overall_binary_dist = _binary_distribution(df_scene_overall[overall_bool_mask])
overall_n_all = int(len(df_scene_overall))
overall_n_binary = int(overall_binary_dist["n"])
overall_n_mcq = int(len(_valid_text_pairs(df_scene_overall[overall_mcq_mask])[0]))
overall_n_text_numeric = int(len(_valid_text_pairs(df_scene_overall[overall_text_numeric_mask])[0]))

overall_ds_overall = {"n": overall_n_all}
overall_ds_binary = {"n": overall_n_binary, "yes": overall_binary_dist["yes"], "no": overall_binary_dist["no"]}
overall_ds_mcq = {"n": overall_n_mcq}
overall_ds_text_numeric = {"n": overall_n_text_numeric}

overall_scene_metrics = {
    "overall accuracy": overall_accuracy,
    "accuracy_mcq": overall_accuracy_mcq,
    "Exact Match": overall_exact_match,
    "Macro-averaged F1 score (token-based)": overall_token_f1,
    "Sentence-BERT similarity": overall_sbert,
    "binary tasks": overall_binary_metrics,
    "dataset characteristics": {
        "overall": overall_ds_overall,
        "binary": overall_ds_binary,
        "mcq": overall_ds_mcq,
        "text_numeric": overall_ds_text_numeric,
    },
}
overall_scene_perception_table = _make_excel_table(
    "Overall (Scene perception)",
    [
        ("overall accuracy", overall_accuracy, overall_ds_overall),
        ("accuracy_mcq", overall_accuracy_mcq, overall_ds_mcq),
        ("Exact Match", overall_exact_match, overall_ds_text_numeric),
        ("Macro-averaged F1 score (token-based)", overall_token_f1, overall_ds_text_numeric),
        ("Sentence-BERT similarity", overall_sbert, overall_ds_text_numeric),
        ("binary_tasks/accuracy_binary", overall_binary_metrics["accuracy_binary"], overall_ds_binary),
        ("binary_tasks/f1_macro", overall_binary_metrics["f1_macro"], overall_ds_binary),
        ("binary_tasks/f1_yes", overall_binary_metrics["f1_yes"], overall_ds_binary),
        ("binary_tasks/f1_no", overall_binary_metrics["f1_no"], overall_ds_binary),
        ("binary_tasks/majority_baseline", overall_binary_metrics["majority_baseline"], overall_ds_binary),
        ("binary_tasks/baseline_f1_macro", overall_binary_metrics["baseline_f1_macro"], overall_ds_binary),
        ("binary_tasks/baseline_f1_yes", overall_binary_metrics["baseline_f1_yes"], overall_ds_binary),
        ("binary_tasks/baseline_f1_no", overall_binary_metrics["baseline_f1_no"], overall_ds_binary),
    ],
    dataset_characteristics=None,
)

CLAUDE_SCENE_PERCEPTION_METRICS = {
    "scene_perception_object": object_metrics,
    "scene_perception_feature": feature_metrics,
    "scene_perception_ocr": ocr_metrics,
    "overall_scene_perception": overall_scene_metrics,
}

CLAUDE_SCENE_PERCEPTION_TABLES = {
    "scene_perception_object_table": scene_perception_object_table,
    "scene_perception_feature_table": scene_perception_feature_table,
    "scene_perception_ocr_table": scene_perception_ocr_table,
    "overall_scene_perception_table": overall_scene_perception_table,
}

In [11]:
print("Feature text/numeric rows:", len(feature_text_df))
print("OCR text/numeric rows:", len(ocr_text_num_df))

display(_debug_text_pairs(feature_text_df, 10))
display(_debug_text_pairs(ocr_text_num_df, 10))

print("Feature Sentence-BERT:", _mean_sbert_similarity(feature_text_df))
print("OCR Sentence-BERT:", _mean_sbert_similarity(ocr_text_num_df))

Feature text/numeric rows: 54
OCR text/numeric rows: 36


,gold,pred
0,tiles,tiles
1,laminate,laminate
2,parquet,parquet
3,tiles,tiles
4,parquet,parquet
5,laminate,laminate
6,reinforced concrete,reinforced concrete
7,galvanized steel,galvanized steel
8,mineral wool,mineral wool
9,reinforced concrete,reinforced concrete


,gold,pred
0,"toilet, 1d26; 975; 1181; 974; cutout 1; 1","1, cutout 1, toilet, 1d26, 1181, 974, 975"
1,"toilet, 1d26; 975",toilet 1d26 975
2,"child bedroom, 14 mВІ; guest bedroom, 24 mВІ; gu...","child bedroom 14 mВІ, guest bedroom 24 mВІ, wc 5..."
3,"child bedroom, 14 mВІ; guest bedroom, 24 mВІ; gu...","child bedroom 14 mВІ, guest bedroom 24 mВІ, wc 5..."
4,"ventilated facade panels, air, reinforced conc...","ventilated facade panels, air, reinforced conc..."
5,"reinforced concrete slab, 200 mm, 2 layers of ...","ceramic tile, 10 mm; bonding layer of tile adh..."
6,mesh-reinforced plaster,"mesh reinforced plaster, insulation, foam conc..."
7,"reinforced concrete slab, 300 mm, waterproofin...","laminate flooring, 12 mm; adhesive compound; s..."
8,"600 mm, 900}};text={""shower curtain mm","2, cutout 6, 1, cutout 6, 600, 1125, 900, 1145..."
9,"600 mm, 900 mm; shower curtain","2, cutout 6, 1, cutout 6, 600, 495, 915, 900, ..."


Feature Sentence-BERT: 0.9060295224189758
OCR Sentence-BERT: 0.8369554281234741


In [12]:
print("Section sizes:")
print("Scene perception (object):", len(df_object))
print("Scene perception (feature):", len(df_feature))
print("Scene perception (OCR):", len(df_ocr))
print("Overall (Scene perception):", len(df_scene_overall))

print("\nExcel-ready tables:")
display(scene_perception_object_table)
display(scene_perception_feature_table)
display(scene_perception_ocr_table)
display(overall_scene_perception_table)

Section sizes:
Scene perception (object): 154
Scene perception (feature): 241
Scene perception (OCR): 60
Overall (Scene perception): 455

Excel-ready tables:


,Table,Metric,Value,Dataset characteristics
0,Scene perception (object),overall accuracy,0.785714,n=154
1,Scene perception (object),accuracy_mcq,NaN,n=0
2,Scene perception (object),binary_tasks/accuracy_binary,0.785714,"n=154, yes=108, no=46"
3,Scene perception (object),binary_tasks/f1_macro,0.748851,"n=154, yes=108, no=46"
4,Scene perception (object),binary_tasks/f1_yes,0.845070,"n=154, yes=108, no=46"
5,Scene perception (object),binary_tasks/f1_no,0.652632,"n=154, yes=108, no=46"
6,Scene perception (object),binary_tasks/majority_baseline,0.701299,"n=154, yes=108, no=46"
7,Scene perception (object),binary_tasks/baseline_f1_macro,0.412214,"n=154, yes=108, no=46"
8,Scene perception (object),binary_tasks/baseline_f1_yes,0.824427,"n=154, yes=108, no=46"
9,Scene perception (object),binary_tasks/baseline_f1_no,0.000000,"n=154, yes=108, no=46"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (feature),overall accuracy,0.622407,n=241
1,Scene perception (feature),Sentence-BERT similarity,0.906030,n=54
2,Scene perception (feature),Macro-averaged F1 score (token-based),0.845679,n=54
3,Scene perception (feature),Exact Match,0.833333,n=54
4,Scene perception (feature),binary_tasks/f1_macro,0.558658,"n=187, yes=69, no=118"
5,Scene perception (feature),binary_tasks/f1_yes,0.594059,"n=187, yes=69, no=118"
6,Scene perception (feature),binary_tasks/f1_no,0.523256,"n=187, yes=69, no=118"
7,Scene perception (feature),binary_tasks/majority_baseline,0.631016,"n=187, yes=69, no=118"
8,Scene perception (feature),binary_tasks/baseline_f1_macro,0.386885,"n=187, yes=69, no=118"
9,Scene perception (feature),binary_tasks/baseline_f1_yes,0.000000,"n=187, yes=69, no=118"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (OCR),Exact Match,0.194444,n=36
1,Scene perception (OCR),Sentence-BERT similarity,0.836955,n=36
2,Scene perception (OCR),Macro-averaged F1 score,0.558924,n=36
3,Scene perception (OCR),accuracy_binary,1.000000,"n=24, yes=24, no=0"
4,Scene perception (OCR),binary_tasks/f1_macro,1.000000,"n=24, yes=24, no=0"
5,Scene perception (OCR),binary_tasks/f1_yes,1.000000,"n=24, yes=24, no=0"
6,Scene perception (OCR),binary_tasks/f1_no,0.000000,"n=24, yes=24, no=0"
7,Scene perception (OCR),binary_tasks/majority_baseline,1.000000,"n=24, yes=24, no=0"
8,Scene perception (OCR),binary_tasks/baseline_f1_macro,1.000000,"n=24, yes=24, no=0"
9,Scene perception (OCR),binary_tasks/baseline_f1_yes,1.000000,"n=24, yes=24, no=0"


,Table,Metric,Value,Dataset characteristics
0,Overall (Scene perception),overall accuracy,0.663736,n=455
1,Overall (Scene perception),accuracy_mcq,NaN,n=0
2,Overall (Scene perception),Exact Match,0.577778,n=90
3,Overall (Scene perception),Macro-averaged F1 score (token-based),0.730977,n=90
4,Overall (Scene perception),Sentence-BERT similarity,0.878400,n=90
5,Overall (Scene perception),binary_tasks/accuracy_binary,0.684932,"n=365, yes=201, no=164"
6,Overall (Scene perception),binary_tasks/f1_macro,0.660454,"n=365, yes=201, no=164"
7,Overall (Scene perception),binary_tasks/f1_yes,0.751620,"n=365, yes=201, no=164"
8,Overall (Scene perception),binary_tasks/f1_no,0.569288,"n=365, yes=201, no=164"
9,Overall (Scene perception),binary_tasks/majority_baseline,0.550685,"n=365, yes=201, no=164"


In [13]:
preview_cols = [
    "relation_type",
    "question_text",
    "correct_answer",
    "model_answer",
    "bool_model_answer",
    "gt_norm",
    "is_correct",
]
show_cols = [c for c in preview_cols if c in df.columns]

if len(show_cols) > 0:
    display(df[show_cols].head(20))
else:
    show_cols = [c for c in ["model_answer", "bool_model_answer"] if c in df.columns]
    display(df[show_cols].head(20))


,question_text,model_answer,bool_model_answer,gt_norm,is_correct
0,"Does this view contain a handrails? \nAnswer ""...",The floor plan shows a toilet room (1D26) with...,yes,yes,True
1,"Does this view contain a toilet? \nAnswer ""yes...","The floor plan shows a room labeled ""TOILET 1D...",yes,yes,True
2,"Does this view contain a sink? \nAnswer ""yes"" ...","The floor plan shows a toilet room (labeled ""T...",yes,yes,True
3,"Does this view contain a door? \nAnswer ""yes"" ...",The floor plan shows two door swings вЂ” one arc...,yes,yes,True
4,"Does this view contain a wall? \nAnswer ""yes"" ...",This is a floor plan drawing of a toilet room ...,yes,yes,True
5,Does this view contain a paper towel dispenser...,The floor plan shows a toilet room (1D26) with...,yes,yes,True
6,Does this view contain a service duct? \nAnswe...,The floor plan shows a toilet room (1D26) with...,yes,yes,True
7,Does this view contain a sanitary fixture? \nA...,"The floor plan shows a toilet room (labeled ""T...",yes,yes,True
8,"Does this view contain a grabrail? \nAnswer ""y...",The floor plan of the toilet room shows grab r...,yes,yes,True
9,Does this view contain a curtain wall? \nAnswe...,This is a floor plan view of a toilet room (1D...,no,no,True


### Confusion matrix (scene-perception sections)

In [14]:
import numpy as np
import pandas as pd

required_sections = ["df_object", "df_feature", "df_ocr", "df_scene_overall"]
missing_sections = [name for name in required_sections if name not in globals()]
if missing_sections:
    raise RuntimeError(
        "Run the scene-perception aggregation cells first. Missing: " + ", ".join(missing_sections)
    )

section_specs = [
    ("scene_perception_object", "Scene perception (object)", df_object),
    ("scene_perception_feature", "Scene perception (feature)", df_feature),
    ("scene_perception_ocr", "Scene perception (OCR)", df_ocr),
    ("overall_scene_perception", "Overall (Scene perception)", df_scene_overall),
]


def _safe_ratio(num: int, den: int) -> float:
    if den == 0:
        return np.nan
    return float(num / den)


def _section_confusion_stats(section_df: pd.DataFrame):
    section_n = int(len(section_df))

    if "gt_norm" in section_df.columns:
        gt_bool = section_df["gt_norm"].map(_norm_bool)
    else:
        gt_bool = section_df[COL_GT].map(_norm_bool)

    gt_binary_n = int(gt_bool.isin(["yes", "no"]).sum())
    ev = _binary_eval_view(section_df)
    evaluable_n = int(len(ev))

    if evaluable_n == 0:
        tp = tn = fp = fn = 0
    else:
        y_true = ev["_gt_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
        y_pred = ev["_pred_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

    satisfied_n = int(tp + tn)

    counts_df = pd.DataFrame(
        [[tp, fn], [fp, tn]],
        index=["actual_yes", "actual_no"],
        columns=["pred_yes", "pred_no"],
    )

    ratios_df = pd.DataFrame(
        [
            [_safe_ratio(tp, evaluable_n), _safe_ratio(fn, evaluable_n)],
            [_safe_ratio(fp, evaluable_n), _safe_ratio(tn, evaluable_n)],
        ],
        index=["actual_yes", "actual_no"],
        columns=["pred_yes", "pred_no"],
    )

    stats = {
        "n_section": section_n,
        "n_binary_gt": gt_binary_n,
        "n_evaluable": evaluable_n,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp_ratio": _safe_ratio(tp, evaluable_n),
        "tn_ratio": _safe_ratio(tn, evaluable_n),
        "satisfied_questions": satisfied_n,
        "satisfied_ratio": _safe_ratio(satisfied_n, evaluable_n),
        "evaluable_ratio_from_binary_gt": _safe_ratio(evaluable_n, gt_binary_n),
    }
    return stats, counts_df, ratios_df


confusion_rows = []
CLAUDE_SCENE_PERCEPTION_CONFUSION = {}

for section_key, section_title, section_df in section_specs:
    stats, counts_df, ratios_df = _section_confusion_stats(section_df)

    confusion_rows.append(
        {
            "Section": section_title,
            "n_section": stats["n_section"],
            "n_binary_gt": stats["n_binary_gt"],
            "n_evaluable": stats["n_evaluable"],
            "TP": stats["tp"],
            "TN": stats["tn"],
            "FP": stats["fp"],
            "FN": stats["fn"],
            "TP_ratio": stats["tp_ratio"],
            "TN_ratio": stats["tn_ratio"],
            "satisfied_questions": stats["satisfied_questions"],
            "satisfied_ratio": stats["satisfied_ratio"],
            "evaluable_ratio_from_binary_gt": stats["evaluable_ratio_from_binary_gt"],
        }
    )

    CLAUDE_SCENE_PERCEPTION_CONFUSION[section_key] = {
        "section": section_title,
        "n_section": stats["n_section"],
        "n_binary_gt": stats["n_binary_gt"],
        "n_evaluable": stats["n_evaluable"],
        "counts": {
            "tp": stats["tp"],
            "tn": stats["tn"],
            "fp": stats["fp"],
            "fn": stats["fn"],
        },
        "ratios": {
            "tp_ratio": stats["tp_ratio"],
            "tn_ratio": stats["tn_ratio"],
            "satisfied_ratio": stats["satisfied_ratio"],
            "evaluable_ratio_from_binary_gt": stats["evaluable_ratio_from_binary_gt"],
        },
        "satisfied_questions": stats["satisfied_questions"],
        "matrix_counts": counts_df,
        "matrix_ratios": ratios_df,
    }

    print(f"\n{section_title}")
    print(
        f"n_section={stats['n_section']} | n_binary_gt={stats['n_binary_gt']} | "
        f"n_evaluable={stats['n_evaluable']} | satisfied={stats['satisfied_questions']}"
    )
    display(counts_df)
    display(ratios_df)

scene_perception_confusion_table = pd.DataFrame(confusion_rows)
display(scene_perception_confusion_table)

if "CLAUDE_SCENE_PERCEPTION_METRICS" in globals() and isinstance(CLAUDE_SCENE_PERCEPTION_METRICS, dict):
    CLAUDE_SCENE_PERCEPTION_METRICS["scene_perception_confusion"] = CLAUDE_SCENE_PERCEPTION_CONFUSION

if "CLAUDE_SCENE_PERCEPTION_TABLES" in globals() and isinstance(CLAUDE_SCENE_PERCEPTION_TABLES, dict):
    CLAUDE_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"] = scene_perception_confusion_table



Scene perception (object)
n_section=154 | n_binary_gt=154 | n_evaluable=154 | satisfied=121


,pred_yes,pred_no
actual_yes,90,18
actual_no,15,31


,pred_yes,pred_no
actual_yes,0.584416,0.116883
actual_no,0.097403,0.201299



Scene perception (feature)
n_section=241 | n_binary_gt=187 | n_evaluable=187 | satisfied=105


,pred_yes,pred_no
actual_yes,60,9
actual_no,73,45


...

,pred_yes,pred_no
actual_yes,1.0,0.0
actual_no,0.0,0.0



Overall (Scene perception)
n_section=455 | n_binary_gt=365 | n_evaluable=365 | satisfied=250


,pred_yes,pred_no
actual_yes,174,27
actual_no,88,76


,pred_yes,pred_no
actual_yes,0.476712,0.073973
actual_no,0.241096,0.208219


,Section,n_section,n_binary_gt,n_evaluable,TP,TN,FP,FN,TP_ratio,TN_ratio,satisfied_questions,satisfied_ratio,evaluable_ratio_from_binary_gt
0,Scene perception (object),154,154,154,90,31,15,18,0.584416,0.201299,121,0.785714,1.0
1,Scene perception (feature),241,187,187,60,45,73,9,0.320856,0.240642,105,0.561497,1.0
2,Scene perception (OCR),60,24,24,24,0,0,0,1.000000,0.000000,24,1.000000,1.0
3,Overall (Scene perception),455,365,365,174,76,88,27,0.476712,0.208219,250,0.684932,1.0


### Save dictionary

In [15]:
if "SKIP_CLAUDE" in globals() and SKIP_CLAUDE:
    print(f"{MODEL_NAME}: missing input CSV. Save dictionary cell skipped.")
    CLAUDE_METRICS = None
    CLAUDE_RELATION_GROUP_METRICS = None
    CLAUDE_RELATION_GROUP_TABLES = None
else:
    if "CLAUDE_SCENE_PERCEPTION_METRICS" not in globals():
        raise RuntimeError(
            "CLAUDE_SCENE_PERCEPTION_METRICS not found. Run the scene-perception aggregation cell first."
        )
    if "CLAUDE_SCENE_PERCEPTION_TABLES" not in globals():
        raise RuntimeError(
            "CLAUDE_SCENE_PERCEPTION_TABLES not found. Run the scene-perception aggregation cell first."
        )

    CLAUDE_METRICS = {
        "Scene perception (object)": CLAUDE_SCENE_PERCEPTION_METRICS["scene_perception_object"],
        "Scene perception (feature)": CLAUDE_SCENE_PERCEPTION_METRICS["scene_perception_feature"],
        "Scene perception (OCR)": CLAUDE_SCENE_PERCEPTION_METRICS["scene_perception_ocr"],
        "Overall (Scene perception)": CLAUDE_SCENE_PERCEPTION_METRICS["overall_scene_perception"],
    }

    if "scene_perception_confusion" in CLAUDE_SCENE_PERCEPTION_METRICS:
        CLAUDE_METRICS["Confusion matrix (Scene perception)"] = CLAUDE_SCENE_PERCEPTION_METRICS["scene_perception_confusion"]

    # Keep variable names for downstream cells that still expect these globals.
    CLAUDE_RELATION_GROUP_METRICS = CLAUDE_METRICS
    CLAUDE_RELATION_GROUP_TABLES = CLAUDE_SCENE_PERCEPTION_TABLES

    if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
        MODEL_METRICS = {}
    MODEL_METRICS[MODEL_NAME] = CLAUDE_METRICS

CLAUDE_RELATION_GROUP_METRICS


{'Scene perception (object)': {'overall accuracy': 0.7857142857142857,
  'accuracy_mcq': nan,
  'binary tasks': {'accuracy_binary': 0.7857142857142857,
   'f1_macro': 0.7488510007412899,
   'f1_yes': 0.8450704225352113,
   'f1_no': 0.6526315789473685,
   'majority_baseline': 0.7012987012987013,
   'baseline_f1_macro': 0.4122137404580153,
   'baseline_f1_yes': 0.8244274809160306,
   'baseline_f1_no': 0.0},
  'dataset characteristics': {'overall': {'n': 154},
   'binary': {'n': 154, 'yes': 108, 'no': 46},
   'mcq': {'n': 0},
   'text_numeric': {'n': 0}}},
 'Scene perception (feature)': {'overall accuracy': 0.6224066390041494,
  'Sentence-BERT similarity': 0.9060295224189758,
  'Macro-averaged F1 score (token-based)': 0.845679012345679,
  'Exact Match': 0.8333333333333334,
  'binary tasks': {'accuracy_binary': 0.5614973262032086,
   'f1_macro': 0.5586576099470413,
   'f1_yes': 0.594059405940594,
   'f1_no': 0.5232558139534884,
   'majority_baseline': 0.6310160427807486,
   'baseline_f1_ma

# 3. Gemini 3 Pro preview

### F1&Accuracy by template

In [16]:
import pandas as pd
import re
from pathlib import Path

MODEL_NAME = "Gemini 3 Pro Preview"
INPUT = "1_1_with_answers_gemini.csv"

COL_TEMPLATE = "template_id"
COL_GT_CANDIDATES = ["ground_truth_answer", "ground_truth", "correct_answer"]
COL_PRED = "model_answer"

SKIP_GEMINI = not Path(INPUT).exists()
if SKIP_GEMINI:
    print(f"{MODEL_NAME}: missing {INPUT}. Cell skipped. Dictionary not saved.")
    df = pd.DataFrame(columns=[COL_TEMPLATE, COL_PRED] + COL_GT_CANDIDATES + ["bool_model_answer"])
else:
    df = pd.read_csv(INPUT)


def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t


def extract_yes_no_from_text(text: str) -> str:
    s = "" if pd.isna(text) else str(text)
    matches = re.findall(r"@\s*(YES|NO)\b", s, flags=re.IGNORECASE)
    if matches:
        return matches[-1].lower()
    t = normalize_text(s)
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    return ""


if not SKIP_GEMINI:
    COL_GT = next((c for c in COL_GT_CANDIDATES if c in df.columns), None)
    if COL_GT is None:
        raise KeyError(f"No GT column found. Tried {COL_GT_CANDIDATES}. Available: {list(df.columns)}")
    if COL_PRED not in df.columns:
        raise KeyError(f"Prediction column '{COL_PRED}' not found. Available: {list(df.columns)}")

    if "bool_model_answer" not in df.columns:
        df["bool_model_answer"] = df[COL_PRED].apply(extract_yes_no_from_text)
    else:
        df["bool_model_answer"] = df["bool_model_answer"].apply(normalize_label)
        missing_mask = df["bool_model_answer"].isin(["", "nan", "none", "null"])
        df.loc[missing_mask, "bool_model_answer"] = df.loc[missing_mask, COL_PRED].apply(extract_yes_no_from_text)

    df["gt_norm"] = df[COL_GT].apply(normalize_label)
    pred_norm = df["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
    pred_norm = pred_norm.where(pred_norm.isin(["yes", "no"]), "")
    df["bool_model_answer"] = pred_norm
    df["is_correct"] = df["bool_model_answer"] == df["gt_norm"]


# ---- template preview report (same structure as Claude section) ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({"n": n, "accuracy": acc, "f1": f1, "f1_type": f1_type})


if SKIP_GEMINI:
    report = pd.DataFrame(columns=["n", "accuracy", "f1", "f1_type", COL_TEMPLATE])
else:
    rows = []
    for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
        m = group_metrics(g)
        m[COL_TEMPLATE] = template_id
        rows.append(m)
    report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)

report


,n,accuracy,f1,f1_type,template_id
0,154,0.922078,0.945455,binary(yes),1
1,187,0.588235,0.579235,binary(yes),2
2,1,0.000000,0.000000,macro(GT-classes),3
3,38,0.000000,0.000000,macro(GT-classes),4
4,6,0.000000,0.000000,macro(GT-classes),5
5,10,0.000000,0.000000,macro(GT-classes),7
6,24,1.000000,1.000000,binary(yes),8
7,18,0.000000,0.000000,macro(GT-classes),9
8,10,0.000000,0.000000,macro(GT-classes),10
9,8,0.000000,0.000000,macro(GT-classes),12


### Save Gemini dictionary

In [17]:
import numpy as np
import pandas as pd
import re

REQUIRED_SCENE_HELPERS = [
    "_binary_metrics",
    "_binary_distribution",
    "_exact_match",
    "_accuracy_mcq",
    "_macro_token_f1",
    "_mean_sbert_similarity",
    "_taskwise_overall_accuracy",
    "_make_excel_table",
    "_binary_eval_view",
    "_norm_bool",
]


def _extract_answer_after_at_local(v) -> str:
    if "_extract_answer_after_at" in globals():
        return _extract_answer_after_at(v)
    s = "" if pd.isna(v) else str(v).strip()
    if not s:
        return ""
    if "@" in s:
        s = s.rsplit("@", 1)[-1].strip()
    s = s.splitlines()[0].strip() if s else ""
    s = s.strip(" \t\n\r-:;,.\"'`()[]{}")
    return s


def _build_scene_metrics_bundle(df_source: pd.DataFrame):
    missing_helpers = [h for h in REQUIRED_SCENE_HELPERS if h not in globals()]
    if missing_helpers:
        raise RuntimeError(
            "Run Claude scene-perception helper cell first. Missing helpers: " + ", ".join(missing_helpers)
        )

    df_eval = df_source.copy()
    df_eval["_template_id_norm"] = pd.to_numeric(df_eval.get("template_id"), errors="coerce").astype("Int64")
    layer_source = "layer_1" if "layer_1" in df_eval.columns else ("layer_id" if "layer_id" in df_eval.columns else None)
    if layer_source is None:
        df_eval["_layer_1_norm"] = ""
    else:
        df_eval["_layer_1_norm"] = (
            df_eval[layer_source].fillna("").astype(str).str.strip().str.replace(".", "_", regex=False)
        )
    df_eval["_answer_type_norm"] = df_eval.get("answer_type", "").fillna("").astype(str).str.strip().str.lower()

    if "bool_model_answer" not in df_eval.columns:
        df_eval["bool_model_answer"] = ""

    parsed = df_eval.get("parsed_model_answer", pd.Series(["" for _ in range(len(df_eval))], index=df_eval.index))
    parsed = parsed.fillna("").astype(str).str.strip()
    parsed_fallback = df_eval[COL_PRED].apply(_extract_answer_after_at_local)
    df_eval["parsed_model_answer"] = parsed.where(parsed != "", parsed_fallback)

    ocr_mask = df_eval["_layer_1_norm"] == "1_1_2"
    object_mask = (df_eval["_template_id_norm"] == 1) & (~ocr_mask)
    feature_mask = df_eval["_template_id_norm"].isin([2, 4, 5, 7]) & (~ocr_mask)

    feature_binary_mask = feature_mask & (df_eval["_template_id_norm"] == 2)
    feature_text_mask = feature_mask & df_eval["_template_id_norm"].isin([4, 5, 7])
    feature_mcq_mask = feature_mask & (~feature_text_mask) & df_eval["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])

    ocr_bool_mask = ocr_mask & (df_eval["_answer_type_norm"] == "bool")
    ocr_text_numeric_mask = ocr_mask & df_eval["_answer_type_norm"].isin(["numeric", "text"])

    df_object = df_eval[object_mask].copy()
    df_feature = df_eval[feature_mask].copy()
    df_ocr = df_eval[ocr_mask].copy()

    overall_idx = df_object.index.union(df_feature.index).union(df_ocr.index)
    df_scene_overall = df_eval.loc[overall_idx].copy()

    object_binary_metrics = _binary_metrics(df_object)
    object_dist = _binary_distribution(df_object)

    object_ds_overall = {"n": int(len(df_object))}
    object_ds_binary = {
        "n": int(object_dist["n"]),
        "yes": object_dist["yes"],
        "no": object_dist["no"],
    }
    object_ds_mcq = {"n": 0}
    object_ds_text_numeric = {"n": 0}

    object_metrics = {
        "overall accuracy": object_binary_metrics["accuracy_binary"],
        "accuracy_mcq": np.nan,
        "binary tasks": object_binary_metrics,
        "dataset characteristics": {
            "overall": object_ds_overall,
            "binary": object_ds_binary,
            "mcq": object_ds_mcq,
            "text_numeric": object_ds_text_numeric,
        },
    }
    scene_perception_object_table = _make_excel_table(
        "Scene perception (object)",
        [
            ("overall accuracy", object_metrics["overall accuracy"], object_ds_overall),
            ("accuracy_mcq", np.nan, object_ds_mcq),
            ("binary_tasks/accuracy_binary", object_binary_metrics["accuracy_binary"], object_ds_binary),
            ("binary_tasks/f1_macro", object_binary_metrics["f1_macro"], object_ds_binary),
            ("binary_tasks/f1_yes", object_binary_metrics["f1_yes"], object_ds_binary),
            ("binary_tasks/f1_no", object_binary_metrics["f1_no"], object_ds_binary),
            ("binary_tasks/majority_baseline", object_binary_metrics["majority_baseline"], object_ds_binary),
            ("binary_tasks/baseline_f1_macro", object_binary_metrics["baseline_f1_macro"], object_ds_binary),
            ("binary_tasks/baseline_f1_yes", object_binary_metrics["baseline_f1_yes"], object_ds_binary),
            ("binary_tasks/baseline_f1_no", object_binary_metrics["baseline_f1_no"], object_ds_binary),
        ],
    )

    feature_binary_df = df_eval[feature_binary_mask].copy()
    feature_text_df = df_eval[feature_text_mask].copy()
    feature_binary_metrics = _binary_metrics(feature_binary_df)
    feature_binary_dist = _binary_distribution(feature_binary_df)
    feature_exact_match = _exact_match(feature_text_df)
    feature_token_f1 = _macro_token_f1(feature_text_df)
    feature_sbert = _mean_sbert_similarity(feature_text_df)
    feature_overall_accuracy = _taskwise_overall_accuracy(
        df_feature,
        feature_binary_mask.loc[df_feature.index],
        feature_mcq_mask.loc[df_feature.index],
        feature_text_mask.loc[df_feature.index],
    )
    feature_ds_overall = {"n": int(len(df_feature))}
    feature_ds_binary = {"n": int(feature_binary_dist["n"]), "yes": feature_binary_dist["yes"], "no": feature_binary_dist["no"]}
    feature_ds_text_numeric = {"n": int(len(_valid_text_pairs(feature_text_df)[0]))}
    feature_metrics = {
        "overall accuracy": feature_overall_accuracy,
        "Sentence-BERT similarity": feature_sbert,
        "Macro-averaged F1 score (token-based)": feature_token_f1,
        "Exact Match": feature_exact_match,
        "binary tasks": feature_binary_metrics,
        "dataset characteristics": {
            "overall": feature_ds_overall,
            "binary": feature_ds_binary,
            "text_numeric": feature_ds_text_numeric,
        },
    }
    scene_perception_feature_table = _make_excel_table(
        "Scene perception (feature)",
        [
            ("overall accuracy", feature_overall_accuracy, feature_ds_overall),
            ("Sentence-BERT similarity", feature_sbert, feature_ds_text_numeric),
            ("Macro-averaged F1 score (token-based)", feature_token_f1, feature_ds_text_numeric),
            ("Exact Match", feature_exact_match, feature_ds_text_numeric),
            ("binary_tasks/f1_macro", feature_binary_metrics["f1_macro"], feature_ds_binary),
            ("binary_tasks/f1_yes", feature_binary_metrics["f1_yes"], feature_ds_binary),
            ("binary_tasks/f1_no", feature_binary_metrics["f1_no"], feature_ds_binary),
            ("binary_tasks/majority_baseline", feature_binary_metrics["majority_baseline"], feature_ds_binary),
            ("binary_tasks/baseline_f1_macro", feature_binary_metrics["baseline_f1_macro"], feature_ds_binary),
            ("binary_tasks/baseline_f1_yes", feature_binary_metrics["baseline_f1_yes"], feature_ds_binary),
            ("binary_tasks/baseline_f1_no", feature_binary_metrics["baseline_f1_no"], feature_ds_binary),
        ],
        dataset_characteristics=None,
    )

    ocr_bool_df = df_eval[ocr_bool_mask].copy()
    ocr_text_num_df = df_eval[ocr_text_numeric_mask].copy()
    ocr_binary_metrics = _binary_metrics(ocr_bool_df)
    ocr_exact_match = _exact_match(ocr_text_num_df)
    ocr_token_f1 = _macro_token_f1(ocr_text_num_df)
    ocr_sbert = _mean_sbert_similarity(ocr_text_num_df)
    ocr_binary_dist = _binary_distribution(ocr_bool_df)
    ocr_ds_binary = {"n": int(ocr_binary_dist["n"]), "yes": ocr_binary_dist["yes"], "no": ocr_binary_dist["no"]}
    ocr_ds_text_numeric = {"n": int(len(_valid_text_pairs(ocr_text_num_df)[0]))}
    ocr_ds_mcq = {"n": int(len(_valid_text_pairs(df_ocr[df_ocr["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])])[0]))}
    ocr_metrics = {
        "Exact Match": ocr_exact_match,
        "Sentence-BERT similarity": ocr_sbert,
        "Macro-averaged F1 score": ocr_token_f1,
        "binary tasks": ocr_binary_metrics,
        "dataset characteristics": {
            "binary": ocr_ds_binary,
            "text_numeric": ocr_ds_text_numeric,
            "mcq": ocr_ds_mcq,
        },
    }
    scene_perception_ocr_table = _make_excel_table(
        "Scene perception (OCR)",
        [
            ("Exact Match", ocr_exact_match, ocr_ds_text_numeric),
            ("Sentence-BERT similarity", ocr_sbert, ocr_ds_text_numeric),
            ("Macro-averaged F1 score", ocr_token_f1, ocr_ds_text_numeric),
            ("accuracy_binary", ocr_binary_metrics["accuracy_binary"], ocr_ds_binary),
            ("binary_tasks/f1_macro", ocr_binary_metrics["f1_macro"], ocr_ds_binary),
            ("binary_tasks/f1_yes", ocr_binary_metrics["f1_yes"], ocr_ds_binary),
            ("binary_tasks/f1_no", ocr_binary_metrics["f1_no"], ocr_ds_binary),
            ("binary_tasks/majority_baseline", ocr_binary_metrics["majority_baseline"], ocr_ds_binary),
            ("binary_tasks/baseline_f1_macro", ocr_binary_metrics["baseline_f1_macro"], ocr_ds_binary),
            ("binary_tasks/baseline_f1_yes", ocr_binary_metrics["baseline_f1_yes"], ocr_ds_binary),
            ("binary_tasks/baseline_f1_no", ocr_binary_metrics["baseline_f1_no"], ocr_ds_binary),
        ],
        dataset_characteristics=None,
    )

    overall_bool_mask = df_scene_overall["_answer_type_norm"] == "bool"
    overall_feature_text_templates_mask = df_scene_overall["_template_id_norm"].isin([4, 5, 7])
    overall_mcq_mask = (~overall_bool_mask) & (~overall_feature_text_templates_mask) & df_scene_overall["_answer_type_norm"].isin(["mcq", "multiple_choice", "multiple choice"])
    overall_text_numeric_mask = (~overall_bool_mask) & (
        overall_feature_text_templates_mask | df_scene_overall["_answer_type_norm"].isin(["text", "numeric"])
    )
    overall_binary_metrics = _binary_metrics(df_scene_overall[overall_bool_mask])
    overall_accuracy_mcq = _accuracy_mcq(df_scene_overall[overall_mcq_mask])
    overall_exact_match = _exact_match(df_scene_overall[overall_text_numeric_mask])
    overall_token_f1 = _macro_token_f1(df_scene_overall[overall_text_numeric_mask])
    overall_sbert = _mean_sbert_similarity(df_scene_overall[overall_text_numeric_mask])
    overall_accuracy = _taskwise_overall_accuracy(df_scene_overall, overall_bool_mask, overall_mcq_mask, overall_text_numeric_mask)
    overall_binary_dist = _binary_distribution(df_scene_overall[overall_bool_mask])
    overall_ds_overall = {"n": int(len(df_scene_overall))}
    overall_ds_binary = {"n": int(overall_binary_dist["n"]), "yes": overall_binary_dist["yes"], "no": overall_binary_dist["no"]}
    overall_ds_mcq = {"n": int(len(_valid_text_pairs(df_scene_overall[overall_mcq_mask])[0]))}
    overall_ds_text_numeric = {"n": int(len(_valid_text_pairs(df_scene_overall[overall_text_numeric_mask])[0]))}
    overall_scene_metrics = {
        "overall accuracy": overall_accuracy,
        "accuracy_mcq": overall_accuracy_mcq,
        "Exact Match": overall_exact_match,
        "Macro-averaged F1 score (token-based)": overall_token_f1,
        "Sentence-BERT similarity": overall_sbert,
        "binary tasks": overall_binary_metrics,
        "dataset characteristics": {
            "overall": overall_ds_overall,
            "binary": overall_ds_binary,
            "mcq": overall_ds_mcq,
            "text_numeric": overall_ds_text_numeric,
        },
    }
    overall_scene_perception_table = _make_excel_table(
        "Overall (Scene perception)",
        [
            ("overall accuracy", overall_accuracy, overall_ds_overall),
            ("accuracy_mcq", overall_accuracy_mcq, overall_ds_mcq),
            ("Exact Match", overall_exact_match, overall_ds_text_numeric),
            ("Macro-averaged F1 score (token-based)", overall_token_f1, overall_ds_text_numeric),
            ("Sentence-BERT similarity", overall_sbert, overall_ds_text_numeric),
            ("binary_tasks/accuracy_binary", overall_binary_metrics["accuracy_binary"], overall_ds_binary),
            ("binary_tasks/f1_macro", overall_binary_metrics["f1_macro"], overall_ds_binary),
            ("binary_tasks/f1_yes", overall_binary_metrics["f1_yes"], overall_ds_binary),
            ("binary_tasks/f1_no", overall_binary_metrics["f1_no"], overall_ds_binary),
            ("binary_tasks/majority_baseline", overall_binary_metrics["majority_baseline"], overall_ds_binary),
            ("binary_tasks/baseline_f1_macro", overall_binary_metrics["baseline_f1_macro"], overall_ds_binary),
            ("binary_tasks/baseline_f1_yes", overall_binary_metrics["baseline_f1_yes"], overall_ds_binary),
            ("binary_tasks/baseline_f1_no", overall_binary_metrics["baseline_f1_no"], overall_ds_binary),
        ],
        dataset_characteristics=None,
    )

    metrics = {
        "scene_perception_object": object_metrics,
        "scene_perception_feature": feature_metrics,
        "scene_perception_ocr": ocr_metrics,
        "overall_scene_perception": overall_scene_metrics,
    }
    tables = {
        "scene_perception_object_table": scene_perception_object_table,
        "scene_perception_feature_table": scene_perception_feature_table,
        "scene_perception_ocr_table": scene_perception_ocr_table,
        "overall_scene_perception_table": overall_scene_perception_table,
    }
    sections = {
        "scene_perception_object": df_object,
        "scene_perception_feature": df_feature,
        "scene_perception_ocr": df_ocr,
        "overall_scene_perception": df_scene_overall,
    }
    return metrics, tables, sections


def _build_scene_confusion_bundle(sections: dict):
    def _safe_ratio(num: int, den: int):
        return np.nan if den == 0 else float(num / den)

    rows = []
    conf = {}
    mapping = [
        ("scene_perception_object", "Scene perception (object)"),
        ("scene_perception_feature", "Scene perception (feature)"),
        ("scene_perception_ocr", "Scene perception (OCR)"),
        ("overall_scene_perception", "Overall (Scene perception)"),
    ]
    for key, title in mapping:
        section_df = sections[key]
        section_n = int(len(section_df))
        gt_bool = section_df["gt_norm"].map(_norm_bool) if "gt_norm" in section_df.columns else section_df[COL_GT].map(_norm_bool)
        gt_binary_n = int(gt_bool.isin(["yes", "no"]).sum())
        ev = _binary_eval_view(section_df)
        evaluable_n = int(len(ev))
        if evaluable_n == 0:
            tp = tn = fp = fn = 0
        else:
            y_true = ev["_gt_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
            y_pred = ev["_pred_bool"].map({"yes": 1, "no": 0}).astype(int).to_numpy()
            tp = int(((y_true == 1) & (y_pred == 1)).sum())
            tn = int(((y_true == 0) & (y_pred == 0)).sum())
            fp = int(((y_true == 0) & (y_pred == 1)).sum())
            fn = int(((y_true == 1) & (y_pred == 0)).sum())
        satisfied = int(tp + tn)
        counts_df = pd.DataFrame([[tp, fn], [fp, tn]], index=["actual_yes", "actual_no"], columns=["pred_yes", "pred_no"])
        ratios_df = pd.DataFrame(
            [[_safe_ratio(tp, evaluable_n), _safe_ratio(fn, evaluable_n)], [_safe_ratio(fp, evaluable_n), _safe_ratio(tn, evaluable_n)]],
            index=["actual_yes", "actual_no"],
            columns=["pred_yes", "pred_no"],
        )
        rows.append({
            "Section": title,
            "n_section": section_n,
            "n_binary_gt": gt_binary_n,
            "n_evaluable": evaluable_n,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP_ratio": _safe_ratio(tp, evaluable_n),
            "TN_ratio": _safe_ratio(tn, evaluable_n),
            "satisfied_questions": satisfied,
            "satisfied_ratio": _safe_ratio(satisfied, evaluable_n),
            "evaluable_ratio_from_binary_gt": _safe_ratio(evaluable_n, gt_binary_n),
        })
        conf[key] = {
            "section": title,
            "n_section": section_n,
            "n_binary_gt": gt_binary_n,
            "n_evaluable": evaluable_n,
            "counts": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
            "ratios": {
                "tp_ratio": _safe_ratio(tp, evaluable_n),
                "tn_ratio": _safe_ratio(tn, evaluable_n),
                "satisfied_ratio": _safe_ratio(satisfied, evaluable_n),
                "evaluable_ratio_from_binary_gt": _safe_ratio(evaluable_n, gt_binary_n),
            },
            "satisfied_questions": satisfied,
            "matrix_counts": counts_df,
            "matrix_ratios": ratios_df,
        }
    return conf, pd.DataFrame(rows)


if "SKIP_GEMINI" in globals() and SKIP_GEMINI:
    print("Gemini 3 Pro Preview: missing input CSV. Save dictionary cell skipped.")
    GEMINI_METRICS = None
    GEMINI_RELATION_GROUP_METRICS = None
    GEMINI_RELATION_GROUP_TABLES = None
    GEMINI_SCENE_PERCEPTION_METRICS = None
    GEMINI_SCENE_PERCEPTION_TABLES = None
    GEMINI_SCENE_PERCEPTION_CONFUSION = None
else:
    GEMINI_SCENE_PERCEPTION_METRICS, GEMINI_SCENE_PERCEPTION_TABLES, GEMINI_SCENE_SECTIONS = _build_scene_metrics_bundle(df)
    GEMINI_SCENE_PERCEPTION_CONFUSION, gemini_scene_confusion_table = _build_scene_confusion_bundle(GEMINI_SCENE_SECTIONS)
    GEMINI_SCENE_PERCEPTION_METRICS["scene_perception_confusion"] = GEMINI_SCENE_PERCEPTION_CONFUSION
    GEMINI_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"] = gemini_scene_confusion_table

    GEMINI_METRICS = {
        "Scene perception (object)": GEMINI_SCENE_PERCEPTION_METRICS["scene_perception_object"],
        "Scene perception (feature)": GEMINI_SCENE_PERCEPTION_METRICS["scene_perception_feature"],
        "Scene perception (OCR)": GEMINI_SCENE_PERCEPTION_METRICS["scene_perception_ocr"],
        "Overall (Scene perception)": GEMINI_SCENE_PERCEPTION_METRICS["overall_scene_perception"],
        "Confusion matrix (Scene perception)": GEMINI_SCENE_PERCEPTION_METRICS["scene_perception_confusion"],
    }

    GEMINI_RELATION_GROUP_METRICS = GEMINI_METRICS
    GEMINI_RELATION_GROUP_TABLES = GEMINI_SCENE_PERCEPTION_TABLES

    if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
        MODEL_METRICS = {}
    MODEL_METRICS["Gemini 3 Pro Preview"] = GEMINI_METRICS

    print("Section sizes:")
    print("Scene perception (object):", len(GEMINI_SCENE_SECTIONS["scene_perception_object"]))
    print("Scene perception (feature):", len(GEMINI_SCENE_SECTIONS["scene_perception_feature"]))
    print("Scene perception (OCR):", len(GEMINI_SCENE_SECTIONS["scene_perception_ocr"]))
    print("Overall (Scene perception):", len(GEMINI_SCENE_SECTIONS["overall_scene_perception"]))

    display(GEMINI_SCENE_PERCEPTION_TABLES["scene_perception_object_table"])
    display(GEMINI_SCENE_PERCEPTION_TABLES["scene_perception_feature_table"])
    display(GEMINI_SCENE_PERCEPTION_TABLES["scene_perception_ocr_table"])
    display(GEMINI_SCENE_PERCEPTION_TABLES["overall_scene_perception_table"])
    display(GEMINI_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"])

GEMINI_RELATION_GROUP_METRICS


Section sizes:
Scene perception (object): 154
Scene perception (feature): 241
Scene perception (OCR): 60
Overall (Scene perception): 455


,Table,Metric,Value,Dataset characteristics
0,Scene perception (object),overall accuracy,0.922078,n=154
1,Scene perception (object),accuracy_mcq,NaN,n=0
2,Scene perception (object),binary_tasks/accuracy_binary,0.922078,"n=154, yes=108, no=46"
3,Scene perception (object),binary_tasks/f1_macro,0.904545,"n=154, yes=108, no=46"
4,Scene perception (object),binary_tasks/f1_yes,0.945455,"n=154, yes=108, no=46"
5,Scene perception (object),binary_tasks/f1_no,0.863636,"n=154, yes=108, no=46"
6,Scene perception (object),binary_tasks/majority_baseline,0.701299,"n=154, yes=108, no=46"
7,Scene perception (object),binary_tasks/baseline_f1_macro,0.412214,"n=154, yes=108, no=46"
8,Scene perception (object),binary_tasks/baseline_f1_yes,0.824427,"n=154, yes=108, no=46"
9,Scene perception (object),binary_tasks/baseline_f1_no,0.000000,"n=154, yes=108, no=46"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (feature),overall accuracy,0.643154,n=241
1,Scene perception (feature),Sentence-BERT similarity,0.894728,n=54
2,Scene perception (feature),Macro-averaged F1 score (token-based),0.833333,n=54
3,Scene perception (feature),Exact Match,0.833333,n=54
4,Scene perception (feature),binary_tasks/f1_macro,0.588047,"n=187, yes=69, no=118"
5,Scene perception (feature),binary_tasks/f1_yes,0.579235,"n=187, yes=69, no=118"
6,Scene perception (feature),binary_tasks/f1_no,0.596859,"n=187, yes=69, no=118"
7,Scene perception (feature),binary_tasks/majority_baseline,0.631016,"n=187, yes=69, no=118"
8,Scene perception (feature),binary_tasks/baseline_f1_macro,0.386885,"n=187, yes=69, no=118"
9,Scene perception (feature),binary_tasks/baseline_f1_yes,0.000000,"n=187, yes=69, no=118"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (OCR),Exact Match,0.277778,n=36
1,Scene perception (OCR),Sentence-BERT similarity,0.867218,n=36
2,Scene perception (OCR),Macro-averaged F1 score,0.631055,n=36
3,Scene perception (OCR),accuracy_binary,1.000000,"n=24, yes=24, no=0"
4,Scene perception (OCR),binary_tasks/f1_macro,1.000000,"n=24, yes=24, no=0"
5,Scene perception (OCR),binary_tasks/f1_yes,1.000000,"n=24, yes=24, no=0"
6,Scene perception (OCR),binary_tasks/f1_no,0.000000,"n=24, yes=24, no=0"
7,Scene perception (OCR),binary_tasks/majority_baseline,1.000000,"n=24, yes=24, no=0"
8,Scene perception (OCR),binary_tasks/baseline_f1_macro,1.000000,"n=24, yes=24, no=0"
9,Scene perception (OCR),binary_tasks/baseline_f1_yes,1.000000,"n=24, yes=24, no=0"


,Table,Metric,Value,Dataset characteristics
0,Overall (Scene perception),overall accuracy,0.727473,n=455
1,Overall (Scene perception),accuracy_mcq,NaN,n=0
2,Overall (Scene perception),Exact Match,0.611111,n=90
3,Overall (Scene perception),Macro-averaged F1 score (token-based),0.752422,n=90
4,Overall (Scene perception),Sentence-BERT similarity,0.883724,n=90
5,Overall (Scene perception),binary_tasks/accuracy_binary,0.756164,"n=365, yes=201, no=164"
6,Overall (Scene perception),binary_tasks/f1_macro,0.741832,"n=365, yes=201, no=164"
7,Overall (Scene perception),binary_tasks/f1_yes,0.802661,"n=365, yes=201, no=164"
8,Overall (Scene perception),binary_tasks/f1_no,0.681004,"n=365, yes=201, no=164"
9,Overall (Scene perception),binary_tasks/majority_baseline,0.550685,"n=365, yes=201, no=164"


,Section,n_section,n_binary_gt,n_evaluable,TP,TN,FP,FN,TP_ratio,TN_ratio,satisfied_questions,satisfied_ratio,evaluable_ratio_from_binary_gt
0,Scene perception (object),154,154,154,104,38,8,4,0.675325,0.246753,142,0.922078,1.0
1,Scene perception (feature),241,187,187,53,57,61,16,0.283422,0.304813,110,0.588235,1.0
2,Scene perception (OCR),60,24,24,24,0,0,0,1.000000,0.000000,24,1.000000,1.0
3,Overall (Scene perception),455,365,365,181,95,69,20,0.495890,0.260274,276,0.756164,1.0


{'Scene perception (object)': {'overall accuracy': 0.922077922077922,
  'accuracy_mcq': nan,
  'binary tasks': {'accuracy_binary': 0.922077922077922,
   'f1_macro': 0.9045454545454545,
   'f1_yes': 0.9454545454545454,
   'f1_no': 0.8636363636363636,
   'majority_baseline': 0.7012987012987013,
   'baseline_f1_macro': 0.4122137404580153,
   'baseline_f1_yes': 0.8244274809160306,
   'baseline_f1_no': 0.0},
  'dataset characteristics': {'overall': {'n': 154},
   'binary': {'n': 154, 'yes': 108, 'no': 46},
   'mcq': {'n': 0},
   'text_numeric': {'n': 0}}},
 'Scene perception (feature)': {'overall accuracy': 0.6431535269709544,
  'Sentence-BERT similarity': 0.8947283625602722,
  'Macro-averaged F1 score (token-based)': 0.8333333333333334,
  'Exact Match': 0.8333333333333334,
  'binary tasks': {'accuracy_binary': 0.5882352941176471,
   'f1_macro': 0.5880468057105255,
   'f1_yes': 0.5792349726775956,
   'f1_no': 0.5968586387434555,
   'majority_baseline': 0.6310160427807486,
   'baseline_f1_ma

# 4. GPT 5.2

### F1&Accuracy by template

In [18]:
import pandas as pd
import re
from pathlib import Path

MODEL_NAME = "GPT-5.2"
INPUT = "1_1_with_answers_gpt-5_2.csv"

COL_TEMPLATE = "template_id"
COL_GT_CANDIDATES = ["ground_truth_answer", "ground_truth", "correct_answer"]
COL_PRED = "model_answer"

SKIP_GPT = not Path(INPUT).exists()
if SKIP_GPT:
    print(f"{MODEL_NAME}: missing {INPUT}. Cell skipped. Dictionary not saved.")
    df = pd.DataFrame(columns=[COL_TEMPLATE, COL_PRED] + COL_GT_CANDIDATES + ["bool_model_answer"])
else:
    df = pd.read_csv(INPUT)


def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_label(s: str) -> str:
    t = normalize_text(s)
    if t in {"yes", "true"}:
        return "yes"
    if t in {"no", "false"}:
        return "no"
    return t


def extract_yes_no_from_text(text: str) -> str:
    s = "" if pd.isna(text) else str(text)
    matches = re.findall(r"@\s*(YES|NO)\b", s, flags=re.IGNORECASE)
    if matches:
        return matches[-1].lower()
    t = normalize_text(s)
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    return ""


if not SKIP_GPT:
    COL_GT = next((c for c in COL_GT_CANDIDATES if c in df.columns), None)
    if COL_GT is None:
        raise KeyError(f"No GT column found. Tried {COL_GT_CANDIDATES}. Available: {list(df.columns)}")
    if COL_PRED not in df.columns:
        raise KeyError(f"Prediction column '{COL_PRED}' not found. Available: {list(df.columns)}")

    if "bool_model_answer" not in df.columns:
        df["bool_model_answer"] = df[COL_PRED].apply(extract_yes_no_from_text)
    else:
        df["bool_model_answer"] = df["bool_model_answer"].apply(normalize_label)
        missing_mask = df["bool_model_answer"].isin(["", "nan", "none", "null"])
        df.loc[missing_mask, "bool_model_answer"] = df.loc[missing_mask, COL_PRED].apply(extract_yes_no_from_text)

    df["gt_norm"] = df[COL_GT].apply(normalize_label)
    pred_norm = df["bool_model_answer"].fillna("").astype(str).str.strip().str.lower()
    pred_norm = pred_norm.where(pred_norm.isin(["yes", "no"]), "")
    df["bool_model_answer"] = pred_norm
    df["is_correct"] = df["bool_model_answer"] == df["gt_norm"]


# ---- template preview report (same structure as Claude section) ----
def group_metrics(g: pd.DataFrame) -> pd.Series:
    n = len(g)
    acc = g["is_correct"].mean() if n else 0.0

    gt_set = set(g["gt_norm"].unique())
    if gt_set.issubset({"yes", "no"}):
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        tp = sum((t == "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fp = sum((t != "yes") and (p == "yes") for t, p in zip(y_true, y_pred))
        fn = sum((t == "yes") and (p != "yes") for t, p in zip(y_true, y_pred))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        f1_type = "binary(yes)"
    else:
        y_true = g["gt_norm"].tolist()
        y_pred = g["bool_model_answer"].tolist()
        f1s = []
        for lab in sorted(gt_set):
            tp = sum((t == lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fp = sum((t != lab) and (p == lab) for t, p in zip(y_true, y_pred))
            fn = sum((t == lab) and (p != lab) for t, p in zip(y_true, y_pred))
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1_lab = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
            f1s.append(f1_lab)
        f1 = sum(f1s) / len(f1s) if f1s else 0.0
        f1_type = "macro(GT-classes)"

    return pd.Series({"n": n, "accuracy": acc, "f1": f1, "f1_type": f1_type})


if SKIP_GPT:
    report = pd.DataFrame(columns=["n", "accuracy", "f1", "f1_type", COL_TEMPLATE])
else:
    rows = []
    for template_id, g in df.groupby(COL_TEMPLATE, dropna=False):
        m = group_metrics(g)
        m[COL_TEMPLATE] = template_id
        rows.append(m)
    report = pd.DataFrame(rows).sort_values(COL_TEMPLATE).reset_index(drop=True)

report


,n,accuracy,f1,f1_type,template_id
0,154,0.785714,0.842105,binary(yes),1
1,187,0.614973,0.571429,binary(yes),2
2,1,0.000000,0.000000,macro(GT-classes),3
3,38,0.000000,0.000000,macro(GT-classes),4
4,6,0.000000,0.000000,macro(GT-classes),5
5,10,0.000000,0.000000,macro(GT-classes),7
6,24,1.000000,1.000000,binary(yes),8
7,18,0.000000,0.000000,macro(GT-classes),9
8,10,0.000000,0.000000,macro(GT-classes),10
9,8,0.000000,0.000000,macro(GT-classes),12


### Save GPT-5.2 dictionary

In [19]:
import pandas as pd

if "SKIP_GPT" in globals() and SKIP_GPT:
    print("GPT-5.2: missing input CSV. Save dictionary cell skipped.")
    GPT_METRICS = None
    GPT_RELATION_GROUP_METRICS = None
    GPT_RELATION_GROUP_TABLES = None
    GPT_SCENE_PERCEPTION_METRICS = None
    GPT_SCENE_PERCEPTION_TABLES = None
    GPT_SCENE_PERCEPTION_CONFUSION = None
else:
    if "_build_scene_metrics_bundle" not in globals() or "_build_scene_confusion_bundle" not in globals():
        raise RuntimeError("Run 'Save Gemini dictionary' cell first to initialize shared scene helper functions.")

    GPT_SCENE_PERCEPTION_METRICS, GPT_SCENE_PERCEPTION_TABLES, GPT_SCENE_SECTIONS = _build_scene_metrics_bundle(df)
    GPT_SCENE_PERCEPTION_CONFUSION, gpt_scene_confusion_table = _build_scene_confusion_bundle(GPT_SCENE_SECTIONS)
    GPT_SCENE_PERCEPTION_METRICS["scene_perception_confusion"] = GPT_SCENE_PERCEPTION_CONFUSION
    GPT_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"] = gpt_scene_confusion_table

    GPT_METRICS = {
        "Scene perception (object)": GPT_SCENE_PERCEPTION_METRICS["scene_perception_object"],
        "Scene perception (feature)": GPT_SCENE_PERCEPTION_METRICS["scene_perception_feature"],
        "Scene perception (OCR)": GPT_SCENE_PERCEPTION_METRICS["scene_perception_ocr"],
        "Overall (Scene perception)": GPT_SCENE_PERCEPTION_METRICS["overall_scene_perception"],
        "Confusion matrix (Scene perception)": GPT_SCENE_PERCEPTION_METRICS["scene_perception_confusion"],
    }

    GPT_RELATION_GROUP_METRICS = GPT_METRICS
    GPT_RELATION_GROUP_TABLES = GPT_SCENE_PERCEPTION_TABLES

    if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
        MODEL_METRICS = {}
    MODEL_METRICS["GPT-5.2"] = GPT_METRICS

    print("Section sizes:")
    print("Scene perception (object):", len(GPT_SCENE_SECTIONS["scene_perception_object"]))
    print("Scene perception (feature):", len(GPT_SCENE_SECTIONS["scene_perception_feature"]))
    print("Scene perception (OCR):", len(GPT_SCENE_SECTIONS["scene_perception_ocr"]))
    print("Overall (Scene perception):", len(GPT_SCENE_SECTIONS["overall_scene_perception"]))

    display(GPT_SCENE_PERCEPTION_TABLES["scene_perception_object_table"])
    display(GPT_SCENE_PERCEPTION_TABLES["scene_perception_feature_table"])
    display(GPT_SCENE_PERCEPTION_TABLES["scene_perception_ocr_table"])
    display(GPT_SCENE_PERCEPTION_TABLES["overall_scene_perception_table"])
    display(GPT_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"])

GPT_RELATION_GROUP_METRICS


Section sizes:
Scene perception (object): 154
Scene perception (feature): 241
Scene perception (OCR): 60
Overall (Scene perception): 455


,Table,Metric,Value,Dataset characteristics
0,Scene perception (object),overall accuracy,0.785714,n=154
1,Scene perception (object),accuracy_mcq,NaN,n=0
2,Scene perception (object),binary_tasks/accuracy_binary,0.785714,"n=154, yes=108, no=46"
3,Scene perception (object),binary_tasks/f1_macro,0.754386,"n=154, yes=108, no=46"
4,Scene perception (object),binary_tasks/f1_yes,0.842105,"n=154, yes=108, no=46"
5,Scene perception (object),binary_tasks/f1_no,0.666667,"n=154, yes=108, no=46"
6,Scene perception (object),binary_tasks/majority_baseline,0.701299,"n=154, yes=108, no=46"
7,Scene perception (object),binary_tasks/baseline_f1_macro,0.412214,"n=154, yes=108, no=46"
8,Scene perception (object),binary_tasks/baseline_f1_yes,0.824427,"n=154, yes=108, no=46"
9,Scene perception (object),binary_tasks/baseline_f1_no,0.000000,"n=154, yes=108, no=46"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (feature),overall accuracy,0.651452,n=241
1,Scene perception (feature),Sentence-BERT similarity,0.876255,n=54
2,Scene perception (feature),Macro-averaged F1 score (token-based),0.787037,n=54
3,Scene perception (feature),Exact Match,0.777778,n=54
4,Scene perception (feature),binary_tasks/f1_macro,0.610957,"n=187, yes=69, no=118"
5,Scene perception (feature),binary_tasks/f1_yes,0.571429,"n=187, yes=69, no=118"
6,Scene perception (feature),binary_tasks/f1_no,0.650485,"n=187, yes=69, no=118"
7,Scene perception (feature),binary_tasks/majority_baseline,0.631016,"n=187, yes=69, no=118"
8,Scene perception (feature),binary_tasks/baseline_f1_macro,0.386885,"n=187, yes=69, no=118"
9,Scene perception (feature),binary_tasks/baseline_f1_yes,0.000000,"n=187, yes=69, no=118"


,Table,Metric,Value,Dataset characteristics
0,Scene perception (OCR),Exact Match,0.194444,n=36
1,Scene perception (OCR),Sentence-BERT similarity,0.825044,n=36
2,Scene perception (OCR),Macro-averaged F1 score,0.542961,n=36
3,Scene perception (OCR),accuracy_binary,1.000000,"n=24, yes=24, no=0"
4,Scene perception (OCR),binary_tasks/f1_macro,1.000000,"n=24, yes=24, no=0"
5,Scene perception (OCR),binary_tasks/f1_yes,1.000000,"n=24, yes=24, no=0"
6,Scene perception (OCR),binary_tasks/f1_no,0.000000,"n=24, yes=24, no=0"
7,Scene perception (OCR),binary_tasks/majority_baseline,1.000000,"n=24, yes=24, no=0"
8,Scene perception (OCR),binary_tasks/baseline_f1_macro,1.000000,"n=24, yes=24, no=0"
9,Scene perception (OCR),binary_tasks/baseline_f1_yes,1.000000,"n=24, yes=24, no=0"


,Table,Metric,Value,Dataset characteristics
0,Overall (Scene perception),overall accuracy,0.679121,n=455
1,Overall (Scene perception),accuracy_mcq,NaN,n=0
2,Overall (Scene perception),Exact Match,0.544444,n=90
3,Overall (Scene perception),Macro-averaged F1 score (token-based),0.689407,n=90
4,Overall (Scene perception),Sentence-BERT similarity,0.855771,n=90
5,Overall (Scene perception),binary_tasks/accuracy_binary,0.712329,"n=365, yes=201, no=164"
6,Overall (Scene perception),binary_tasks/f1_macro,0.704339,"n=365, yes=201, no=164"
7,Overall (Scene perception),binary_tasks/f1_yes,0.752941,"n=365, yes=201, no=164"
8,Overall (Scene perception),binary_tasks/f1_no,0.655738,"n=365, yes=201, no=164"
9,Overall (Scene perception),binary_tasks/majority_baseline,0.550685,"n=365, yes=201, no=164"


,Section,n_section,n_binary_gt,n_evaluable,TP,TN,FP,FN,TP_ratio,TN_ratio,satisfied_questions,satisfied_ratio,evaluable_ratio_from_binary_gt
0,Scene perception (object),154,154,154,88,33,13,20,0.571429,0.214286,121,0.785714,1.0
1,Scene perception (feature),241,187,187,48,67,51,21,0.256684,0.358289,115,0.614973,1.0
2,Scene perception (OCR),60,24,24,24,0,0,0,1.000000,0.000000,24,1.000000,1.0
3,Overall (Scene perception),455,365,365,160,100,64,41,0.438356,0.273973,260,0.712329,1.0


{'Scene perception (object)': {'overall accuracy': 0.7857142857142857,
  'accuracy_mcq': nan,
  'binary tasks': {'accuracy_binary': 0.7857142857142857,
   'f1_macro': 0.7543859649122806,
   'f1_yes': 0.8421052631578947,
   'f1_no': 0.6666666666666666,
   'majority_baseline': 0.7012987012987013,
   'baseline_f1_macro': 0.4122137404580153,
   'baseline_f1_yes': 0.8244274809160306,
   'baseline_f1_no': 0.0},
  'dataset characteristics': {'overall': {'n': 154},
   'binary': {'n': 154, 'yes': 108, 'no': 46},
   'mcq': {'n': 0},
   'text_numeric': {'n': 0}}},
 'Scene perception (feature)': {'overall accuracy': 0.6514522821576764,
  'Sentence-BERT similarity': 0.8762553334236145,
  'Macro-averaged F1 score (token-based)': 0.7870370370370371,
  'Exact Match': 0.7777777777777778,
  'binary tasks': {'accuracy_binary': 0.6149732620320856,
   'f1_macro': 0.6109570041608876,
   'f1_yes': 0.5714285714285714,
   'f1_no': 0.6504854368932039,
   'majority_baseline': 0.6310160427807486,
   'baseline_f1_

In [20]:
if "SKIP_GPT" in globals() and SKIP_GPT:
    print("GPT-5.2: missing input CSV. Confusion display skipped.")
else:
    if "GPT_SCENE_PERCEPTION_TABLES" not in globals() or GPT_SCENE_PERCEPTION_TABLES is None:
        raise RuntimeError("Run 'Save GPT-5.2 dictionary' cell first.")
    display(GPT_SCENE_PERCEPTION_TABLES["scene_perception_confusion_table"])
    GPT_SCENE_PERCEPTION_CONFUSION


,Section,n_section,n_binary_gt,n_evaluable,TP,TN,FP,FN,TP_ratio,TN_ratio,satisfied_questions,satisfied_ratio,evaluable_ratio_from_binary_gt
0,Scene perception (object),154,154,154,88,33,13,20,0.571429,0.214286,121,0.785714,1.0
1,Scene perception (feature),241,187,187,48,67,51,21,0.256684,0.358289,115,0.614973,1.0
2,Scene perception (OCR),60,24,24,24,0,0,0,1.000000,0.000000,24,1.000000,1.0
3,Overall (Scene perception),455,365,365,160,100,64,41,0.438356,0.273973,260,0.712329,1.0


# 5. Final table

In [21]:
import pandas as pd

OUTPUT_SUMMARY_CSV = "1_1_scene_perception_summary.csv"

MODEL_COLUMNS = [
    "Claude Opus 4.6",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "GPT-5.4",
]

EXPORT_COLUMNS = ["Table", "Metric"] + MODEL_COLUMNS + ["Dataset characteristics"]

SECTION_TITLE_TO_KEY = {
    "Scene perception (object)": "scene_perception_object",
    "Scene perception (feature)": "scene_perception_feature",
    "Scene perception (OCR)": "scene_perception_ocr",
    "Overall (Scene perception)": "overall_scene_perception",
}

MODEL_VAR_CANDIDATES = {
    "Claude Opus 4.6": ["CLAUDE_METRICS", "CLAUDE_RELATION_GROUP_METRICS"],
    "Gemini 3 Pro Preview": ["GEMINI_METRICS", "GEMINI_RELATION_GROUP_METRICS"],
    "GPT-5.2": ["GPT_METRICS", "GPT_RELATION_GROUP_METRICS"],
    "GPT-5.4": ["GPT54_METRICS", "GPT54_RELATION_GROUP_METRICS", "GPT_5_4_METRICS", "GPT_5_4_RELATION_GROUP_METRICS"],
}

TABLE_DEFS = {
    "Scene perception (object)": [
        {"type": "metric", "label": "overall accuracy", "source": "direct", "key": "overall accuracy", "subset": "overall"},
        {"type": "metric", "label": "accuracy_mcq", "source": "direct", "key": "accuracy_mcq", "subset": "mcq"},
        {"type": "separator", "label": "binary tasks"},
        {"type": "metric", "label": "accuracy_binary", "source": "binary", "key": "accuracy_binary", "subset": "binary"},
        {"type": "metric", "label": "f1_macro", "source": "binary", "key": "f1_macro", "subset": "binary"},
        {"type": "metric", "label": "f1_yes", "source": "binary", "key": "f1_yes", "subset": "binary"},
        {"type": "metric", "label": "f1_no", "source": "binary", "key": "f1_no", "subset": "binary"},
        {"type": "metric", "label": "majority_baseline", "source": "binary", "key": "majority_baseline", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_macro", "source": "binary", "key": "baseline_f1_macro", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_yes", "source": "binary", "key": "baseline_f1_yes", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_no", "source": "binary", "key": "baseline_f1_no", "subset": "binary"},
    ],
    "Scene perception (feature)": [
        {"type": "metric", "label": "overall accuracy", "source": "direct", "key": "overall accuracy", "subset": "overall"},
        {"type": "metric", "label": "accuracy_mcq", "source": "direct", "key": "accuracy_mcq", "subset": "mcq"},
        {"type": "separator", "label": "short text answers"},
        {"type": "metric", "label": "Sentence-BERT similarity", "source": "direct", "key": "Sentence-BERT similarity", "subset": "text_numeric"},
        {"type": "metric", "label": "Macro-averaged F1 score (token-based)", "source": "direct", "key": "Macro-averaged F1 score (token-based)", "subset": "text_numeric"},
        {"type": "metric", "label": "Exact Match", "source": "direct", "key": "Exact Match", "subset": "text_numeric"},
        {"type": "separator", "label": "binary tasks"},
        {"type": "metric", "label": "accuracy_binary", "source": "binary", "key": "accuracy_binary", "subset": "binary"},
        {"type": "metric", "label": "f1_macro", "source": "binary", "key": "f1_macro", "subset": "binary"},
        {"type": "metric", "label": "f1_yes", "source": "binary", "key": "f1_yes", "subset": "binary"},
        {"type": "metric", "label": "f1_no", "source": "binary", "key": "f1_no", "subset": "binary"},
        {"type": "metric", "label": "majority_baseline", "source": "binary", "key": "majority_baseline", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_macro", "source": "binary", "key": "baseline_f1_macro", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_yes", "source": "binary", "key": "baseline_f1_yes", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_no", "source": "binary", "key": "baseline_f1_no", "subset": "binary"},
    ],
    "Scene perception (OCR)": [
        {"type": "metric", "label": "overall accuracy", "source": "direct", "key": "overall accuracy", "subset": "overall"},
        {"type": "separator", "label": "short text answers"},
        {"type": "metric", "label": "Exact Match", "source": "direct", "key": "Exact Match", "subset": "text_numeric"},
        {"type": "metric", "label": "Sentence-BERT similarity", "source": "direct", "key": "Sentence-BERT similarity", "subset": "text_numeric"},
        {"type": "metric", "label": "Macro-averaged F1 score", "source": "direct", "key": "Macro-averaged F1 score", "subset": "text_numeric"},
        {"type": "separator", "label": "binary tasks"},
        {"type": "metric", "label": "accuracy_binary", "source": "binary", "key": "accuracy_binary", "subset": "binary"},
        {"type": "metric", "label": "f1_macro", "source": "binary", "key": "f1_macro", "subset": "binary"},
        {"type": "metric", "label": "f1_yes", "source": "binary", "key": "f1_yes", "subset": "binary"},
        {"type": "metric", "label": "f1_no", "source": "binary", "key": "f1_no", "subset": "binary"},
        {"type": "metric", "label": "majority_baseline", "source": "binary", "key": "majority_baseline", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_macro", "source": "binary", "key": "baseline_f1_macro", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_yes", "source": "binary", "key": "baseline_f1_yes", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_no", "source": "binary", "key": "baseline_f1_no", "subset": "binary"},
    ],
    "Overall (Scene perception)": [
        {"type": "metric", "label": "overall accuracy", "source": "direct", "key": "overall accuracy", "subset": "overall"},
        {"type": "metric", "label": "accuracy_mcq", "source": "direct", "key": "accuracy_mcq", "subset": "mcq"},
        {"type": "separator", "label": "short text answers"},
        {"type": "metric", "label": "Sentence-BERT similarity", "source": "direct", "key": "Sentence-BERT similarity", "subset": "text_numeric"},
        {"type": "metric", "label": "Macro-averaged F1 score (token-based)", "source": "direct", "key": "Macro-averaged F1 score (token-based)", "subset": "text_numeric"},
        {"type": "metric", "label": "Exact Match", "source": "direct", "key": "Exact Match", "subset": "text_numeric"},
        {"type": "separator", "label": "binary tasks"},
        {"type": "metric", "label": "accuracy_binary", "source": "binary", "key": "accuracy_binary", "subset": "binary"},
        {"type": "metric", "label": "f1_macro", "source": "binary", "key": "f1_macro", "subset": "binary"},
        {"type": "metric", "label": "f1_yes", "source": "binary", "key": "f1_yes", "subset": "binary"},
        {"type": "metric", "label": "f1_no", "source": "binary", "key": "f1_no", "subset": "binary"},
        {"type": "metric", "label": "majority_baseline", "source": "binary", "key": "majority_baseline", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_macro", "source": "binary", "key": "baseline_f1_macro", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_yes", "source": "binary", "key": "baseline_f1_yes", "subset": "binary"},
        {"type": "metric", "label": "baseline_f1_no", "source": "binary", "key": "baseline_f1_no", "subset": "binary"},
    ],
}


def _is_blank(v) -> bool:
    if v is None:
        return True
    try:
        return bool(pd.isna(v))
    except Exception:
        pass
    return isinstance(v, str) and v.strip() == ""


def _clean(v):
    return "" if _is_blank(v) else v


def _get_model_scene_metrics(model_name: str):
    if "MODEL_METRICS" in globals() and isinstance(MODEL_METRICS, dict):
        mm = MODEL_METRICS.get(model_name)
        if isinstance(mm, dict):
            return mm
    for var_name in MODEL_VAR_CANDIDATES.get(model_name, []):
        if var_name in globals() and isinstance(globals()[var_name], dict):
            return globals()[var_name]
    return None


def _to_nested_ds(ds):
    if not isinstance(ds, dict):
        return {}
    if any(k in ds for k in ["overall", "binary", "text_numeric", "mcq"]):
        return ds
    return {
        "overall": {"n": ds.get("n")},
        "binary": {"n": ds.get("n"), "yes": ds.get("yes"), "no": ds.get("no")},
        "text_numeric": {"n": 0},
        "mcq": {"n": 0},
    }


def _get_section_block(model_metrics: dict, section_title: str):
    if not isinstance(model_metrics, dict):
        return None
    block = model_metrics.get(section_title)
    return block if isinstance(block, dict) else None


def _get_conf_section(model_metrics: dict, section_title: str):
    if not isinstance(model_metrics, dict):
        return None
    conf = model_metrics.get("Confusion matrix (Scene perception)")
    if not isinstance(conf, dict):
        return None
    section_key = SECTION_TITLE_TO_KEY.get(section_title)
    sec = conf.get(section_key)
    return sec if isinstance(sec, dict) else None


def _get_subset_stats(model_metrics: dict, section_title: str, subset: str):
    block = _get_section_block(model_metrics, section_title)
    if block is None:
        return None
    ds = _to_nested_ds(block.get("dataset characteristics"))
    conf_sec = _get_conf_section(model_metrics, section_title)

    if subset == "overall":
        n = ds.get("overall", {}).get("n")
        if _is_blank(n) and isinstance(conf_sec, dict):
            n = conf_sec.get("n_section")
        return {"n": n}

    if subset == "mcq":
        n = ds.get("mcq", {}).get("n", 0)
        return {"n": n}

    if subset == "text_numeric":
        n = ds.get("text_numeric", {}).get("n", 0)
        return {"n": n}

    if subset == "binary":
        b = ds.get("binary", {})
        n = b.get("n")
        if _is_blank(n) and isinstance(conf_sec, dict):
            n = conf_sec.get("n_binary_gt")
        return {"n": n, "yes": b.get("yes"), "no": b.get("no")}

    return None


def _format_subset_stats(stats: dict, subset: str) -> str:
    if not isinstance(stats, dict):
        return ""
    n = stats.get("n")
    if _is_blank(n):
        return ""
    if subset == "binary":
        yes = stats.get("yes")
        no = stats.get("no")
        if _is_blank(yes) and _is_blank(no):
            return f"n={int(n)}"
        return f"n={int(n)}, yes={'' if _is_blank(yes) else int(yes)}, no={'' if _is_blank(no) else int(no)}"
    return f"n={int(n)}"


def _to_float_or_none(v):
    if _is_blank(v):
        return None
    try:
        x = float(v)
    except Exception:
        return None
    return None if pd.isna(x) else x


def _fallback_overall_accuracy(section_block: dict, section_title: str, model_metrics: dict):
    binary_acc = _to_float_or_none(section_block.get("binary tasks", {}).get("accuracy_binary"))
    text_acc = _to_float_or_none(section_block.get("Exact Match"))
    mcq_acc = _to_float_or_none(section_block.get("accuracy_mcq"))

    b_stats = _get_subset_stats(model_metrics, section_title, "binary") or {}
    t_stats = _get_subset_stats(model_metrics, section_title, "text_numeric") or {}
    m_stats = _get_subset_stats(model_metrics, section_title, "mcq") or {}

    components = []
    for acc, n in [(binary_acc, b_stats.get("n")), (text_acc, t_stats.get("n")), (mcq_acc, m_stats.get("n"))]:
        if acc is None or _is_blank(n):
            continue
        n_int = int(n)
        if n_int <= 0:
            continue
        components.append((acc, n_int))

    if not components:
        return ""
    total_n = sum(n for _, n in components)
    return sum(acc * n for acc, n in components) / total_n if total_n > 0 else ""


def _get_metric_value(model_metrics: dict, section_title: str, row_spec: dict):
    block = _get_section_block(model_metrics, section_title)
    if block is None:
        return ""

    if row_spec["source"] == "binary":
        val = block.get("binary tasks", {}).get(row_spec["key"], "")
    else:
        val = block.get(row_spec["key"], "")

    if row_spec["key"] == "overall accuracy" and _is_blank(val):
        val = _fallback_overall_accuracy(block, section_title, model_metrics)

    return _clean(val)


model_metrics_map = {name: _get_model_scene_metrics(name) for name in MODEL_COLUMNS}

rows = []
for table_title, row_specs in TABLE_DEFS.items():
    for spec in row_specs:
        if spec["type"] == "separator":
            row = {"Table": table_title, "Metric": spec["label"], "Dataset characteristics": ""}
            for model_name in MODEL_COLUMNS:
                row[model_name] = ""
            rows.append(row)
            continue

        dataset_stats = None
        for model_name in MODEL_COLUMNS:
            dataset_stats = _get_subset_stats(model_metrics_map.get(model_name), table_title, spec["subset"])
            if isinstance(dataset_stats, dict) and not _is_blank(dataset_stats.get("n")):
                break
        dataset_text = _format_subset_stats(dataset_stats, spec["subset"])

        row = {
            "Table": table_title,
            "Metric": spec["label"],
            "Dataset characteristics": dataset_text,
        }
        for model_name in MODEL_COLUMNS:
            row[model_name] = _get_metric_value(model_metrics_map.get(model_name), table_title, spec)
        rows.append(row)

summary_df = pd.DataFrame(rows, columns=EXPORT_COLUMNS)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)
print("Saved:", OUTPUT_SUMMARY_CSV)
summary_df


Saved: 1_1_scene_perception_summary.csv


,Table,Metric,Claude Opus 4.6,Gemini 3 Pro Preview,GPT-5.2,GPT-5.4,Dataset characteristics
0,Scene perception (object),overall accuracy,0.785714,0.922078,0.785714,,n=154
1,Scene perception (object),accuracy_mcq,,,,,n=0
2,Scene perception (object),binary tasks,,,,,
3,Scene perception (object),accuracy_binary,0.785714,0.922078,0.785714,,"n=154, yes=108, no=46"
4,Scene perception (object),f1_macro,0.748851,0.904545,0.754386,,"n=154, yes=108, no=46"
5,Scene perception (object),f1_yes,0.84507,0.945455,0.842105,,"n=154, yes=108, no=46"
6,Scene perception (object),f1_no,0.652632,0.863636,0.666667,,"n=154, yes=108, no=46"
7,Scene perception (object),majority_baseline,0.701299,0.701299,0.701299,,"n=154, yes=108, no=46"
8,Scene perception (object),baseline_f1_macro,0.412214,0.412214,0.412214,,"n=154, yes=108, no=46"
9,Scene perception (object),baseline_f1_yes,0.824427,0.824427,0.824427,,"n=154, yes=108, no=46"


### Generate excel

In [23]:
import pandas as pd
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter

csv_path = Path("1_1_scene_perception_summary.csv")
xlsx_path = csv_path.with_suffix(".xlsx")

df = pd.read_csv(csv_path)

required_columns = [
    "Table",
    "Metric",
    "Claude Opus 4.6",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "GPT-5.4",
    "Dataset characteristics",
]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

df = df[required_columns].copy()

BLOCK_ORDER = [
    "Scene perception (object)",
    "Scene perception (feature)",
    "Scene perception (OCR)",
    "Overall (Scene perception)",
]

headers = [
    "Metric",
    "Claude Opus 4.6",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "GPT-5.4",
    "Dataset characteristics",
]

separator_labels = {"short text answers", "binary tasks"}

thin = Side(style="thin", color="000000")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

title_fill = PatternFill("solid", fgColor="D9EAF7")
header_fill = PatternFill("solid", fgColor="EDEDED")
separator_fill = PatternFill("solid", fgColor="F7F7F7")

title_font = Font(bold=True, size=12)
header_font = Font(bold=True)
separator_font = Font(bold=True)
normal_font = Font(bold=False)

center = Alignment(horizontal="center", vertical="center")
center_wrap = Alignment(horizontal="center", vertical="center", wrap_text=True)
left_wrap = Alignment(horizontal="left", vertical="center", wrap_text=True)


def _table_height(block_df: pd.DataFrame) -> int:
    # title + header + all data rows
    return 2 + len(block_df)


def _write_block(ws, start_row, start_col, title, block_df):
    width = len(headers)

    ws.merge_cells(
        start_row=start_row,
        start_column=start_col,
        end_row=start_row,
        end_column=start_col + width - 1,
    )
    title_cell = ws.cell(row=start_row, column=start_col, value=title)
    title_cell.font = title_font
    title_cell.alignment = center
    for c in range(start_col, start_col + width):
        cell = ws.cell(row=start_row, column=c)
        cell.fill = title_fill
        cell.border = border

    hdr_row = start_row + 1
    for j, h in enumerate(headers):
        cell = ws.cell(row=hdr_row, column=start_col + j, value=h)
        cell.font = header_font
        cell.alignment = center_wrap
        cell.fill = header_fill
        cell.border = border

    cur_row = hdr_row + 1
    for _, row in block_df.iterrows():
        metric_name = "" if pd.isna(row.get("Metric")) else str(row.get("Metric")).strip()
        if metric_name.lower() in separator_labels:
            ws.merge_cells(
                start_row=cur_row,
                start_column=start_col,
                end_row=cur_row,
                end_column=start_col + width - 1,
            )
            sub_cell = ws.cell(row=cur_row, column=start_col, value=metric_name)
            sub_cell.font = separator_font
            sub_cell.alignment = left_wrap
            for c in range(start_col, start_col + width):
                cell = ws.cell(row=cur_row, column=c)
                cell.fill = separator_fill
                cell.border = border
            cur_row += 1
            continue

        values = [
            metric_name,
            row.get("Claude Opus 4.6", ""),
            row.get("Gemini 3 Pro Preview", ""),
            row.get("GPT-5.2", ""),
            row.get("GPT-5.4", ""),
            row.get("Dataset characteristics", ""),
        ]
        for j, v in enumerate(values):
            cell = ws.cell(row=cur_row, column=start_col + j, value=v)
            cell.font = normal_font
            cell.alignment = left_wrap if j in [0, 5] else center_wrap
            cell.border = border
        cur_row += 1


blocks = {title: df[df["Table"].astype(str) == title].copy() for title in BLOCK_ORDER}

top_left = BLOCK_ORDER[0]
top_right = BLOCK_ORDER[1]
bottom_left = BLOCK_ORDER[2]
bottom_right = BLOCK_ORDER[3]

top_height = max(_table_height(blocks[top_left]), _table_height(blocks[top_right]))
bottom_start_row = 1 + top_height + 2

BLOCK_POSITIONS = {
    top_left: (1, 1),
    top_right: (1, 9),
    bottom_left: (bottom_start_row, 1),
    bottom_right: (bottom_start_row, 9),
}

wb = Workbook()
ws = wb.active
ws.title = "Scene Perception Summary"

for table_name in BLOCK_ORDER:
    start_row, start_col = BLOCK_POSITIONS[table_name]
    _write_block(ws, start_row, start_col, table_name, blocks[table_name])

for start_col in [1, 9]:
    widths = [34, 16, 22, 12, 12, 34]
    for offset, width in enumerate(widths):
        ws.column_dimensions[get_column_letter(start_col + offset)].width = width

for r in range(1, ws.max_row + 1):
    ws.row_dimensions[r].height = 20

wb.save(xlsx_path)
print(f"Excel file created: {xlsx_path}")


Excel file created: 1_1_scene_perception_summary.xlsx
